# Bikeablity testing

Using this cell to read in some results for one place/scenario at a time, and investigate the different ways of computing bikeablity, along with a few sanity checks. I'm being lazy and reading the results from hard-coded file paths, so you'll need to adjust these.

## Setup

In [ ]:
import pickle
import os
import glob
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import pandas as pd

## Sanity checks 

We load the first result of the "demand" growth order, as it is the most important result for sanity checking. This is because it should have selected the connection to make with the highest possible demand at that stage. We save as geopackages to use in QGIS or similar.

In [ ]:
pickle_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\newcastle\current_ltn_scenario\newcastle_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle"

with open(pickle_path, "rb") as f:
    data = pickle.load(f)

G_full = data["GTs"][1]
G_abs  = data["GT_abstracts"][1]

# Fix missing CRS in abstract graph
if "crs" not in G_abs.graph:
    G_abs.graph["crs"] = "EPSG:3857"

# Convert both graphs
nodes_full, edges_full = ox.graph_to_gdfs(G_full)
nodes_abs,  edges_abs  = ox.graph_to_gdfs(G_abs)

# Output paths
out_full = r"H:\Other\first_demand_full.gpkg"
out_abs  = r"H:\Other\first_demand_abstract.gpkg"


# Save full graph
nodes_full.to_file(out_full, layer="nodes", driver="GPKG")
edges_full.to_file(out_full, layer="edges", driver="GPKG")

# Save abstract graph
nodes_abs.to_file(out_abs, layer="nodes", driver="GPKG")
edges_abs.to_file(out_abs, layer="edges", driver="GPKG")

print("Saved:", out_full)
print("Saved:", out_abs)


edges_abs.plot()

- we can also get first iteration from the the random results to look at in QGIS or similar...

In [ ]:
input_folder = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\newcastle\current_ltn_scenario"

# Folder where you want to save the GeoPackages
output_folder = r"H:\Other"



pattern = os.path.join(input_folder, "newcastle_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle")
files = sorted(glob.glob(pattern))
print(f"Found {len(files)} files")

# for fpath in files:
#     # Extract run number from filename
#     fname = os.path.basename(fpath)
#     run_id = fname.split("_run")[-1].split(".")[0]   # e.g. "00", "01", ...

#     print(f"Processing run {run_id}...")

#     # Load pickle
#     with open(fpath, "rb") as f:
#         data = pickle.load(f)

#     # Extract graphs
#     G_full = data["GTs"][1]            # or [0] if you want the first
#     G_abs  = data["GT_abstracts"][1]   # or [0]

#     # Fix missing CRS in abstract graph
#     if "crs" not in G_abs.graph:
#         G_abs.graph["crs"] = "EPSG:3857"

#     # Convert to GeoDataFrames
#     nodes_full, edges_full = ox.graph_to_gdfs(G_full)
#     nodes_abs,  edges_abs  = ox.graph_to_gdfs(G_abs)

#     # Output paths
#     out_full = os.path.join(output_folder, f"random_run{run_id}_full.gpkg")
#     out_abs  = os.path.join(output_folder, f"random_run{run_id}_abstract.gpkg")

#     # Save full graph
#     nodes_full.to_file(out_full, layer="nodes", driver="GPKG")
#     edges_full.to_file(out_full, layer="edges", driver="GPKG")

#     # Save abstract graph
#     nodes_abs.to_file(out_abs, layer="nodes", driver="GPKG")
#     edges_abs.to_file(out_abs, layer="edges", driver="GPKG")

#     print(f"Saved run {run_id} to:")
#     print("  ", out_full)
#     print("  ", out_abs)

print("All runs processed.")


- we access the demand OD matrix which out trips are based on. the "total_flow" column is the number of trips

In [ ]:
gpkg_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\newcastle\current_ltn_scenario\newcastle_current_ltn_scenario_greedy_demand_weighted.gpkg"
mydemand = gpd.read_file(gpkg_path)
mydemand.head()

- map the demand from mydemand onto the abstract edges we have from above to check what sort of number of trips to expect.

In [ ]:
# Create a merge key in both tables
mydemand["edge_id"] = mydemand.apply(lambda r: tuple(sorted((r.start_osmid, r.end_osmid))), axis=1)
edges_abs = edges_abs.reset_index()  # bring u, v, key into columns
edges_abs["edge_id"] = edges_abs.apply(lambda r: tuple(sorted((r.u, r.v))), axis=1)

# Merge total_flow onto edges_abs 
edges_abs = edges_abs.merge(mydemand[["edge_id", "total_flow"]], on="edge_id",how="left")

# edges_abs now contains a new column: total_flow
edges_abs.head()

# print how many trips we've got just on the first iteration
total_flow_sum = edges_abs["total_flow"].sum()
total_flow_sum

## Super simple bikeablity

The first way to try measureing bikeablity is to just measure each link's number of trips as it gets added. This doesn't consider that other routes might be available, or that some trips could already be done just fine. It does however work as a sanity check to see that the demand approach is 100% better than any random run. 

This first cell just does the first run from demand and random (you can skip this)

In [ ]:
# Path to the reference demand GPKG (contains 'total_flow' per edge)
ref_gpkg_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\newcastle\current_ltn_scenario\newcastle_current_ltn_scenario_greedy_demand_weighted.gpkg"
print("Loading reference demand data...")
mydemand = gpd.read_file(ref_gpkg_path)

# Create merge key 
mydemand["edge_id"] = mydemand.apply(lambda r: tuple(sorted((r.start_osmid, r.end_osmid))), axis=1)
# Keep only what we need for merging to speed things up
reference_flow = mydemand[["edge_id", "total_flow"]]

# Output folder for GPKGs
output_folder = r"H:\Other"

# Lists to store plot data
plot_data = {
    "demand": {"flow": 0, "length": 0},
    "random": {"flow": [], "length": []}
}

def process_edges(edges_gdf, ref_df):
    """Helper to merge flow and calc stats"""
    edges_gdf = edges_gdf.copy()
    edges_gdf = edges_gdf.reset_index()
    edges_gdf["edge_id"] = edges_gdf.apply(lambda r: tuple(sorted((r.u, r.v))), axis=1)
    merged = edges_gdf.merge(ref_df, on="edge_id", how="left")
    total_len = merged.length.sum()
    total_flow = merged["total_flow"].sum()
    
    return total_len, total_flow


demand_pickle_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\newcastle\current_ltn_scenario\newcastle_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle"
print(f"Processing DEMAND scenario...")
with open(demand_pickle_path, "rb") as f:
    data = pickle.load(f)
G_abs = data["GT_abstracts"][1]
if "crs" not in G_abs.graph: G_abs.graph["crs"] = "EPSG:3857"
nodes_abs, edges_abs = ox.graph_to_gdfs(G_abs)

# save for inspection 
out_abs = os.path.join(output_folder, "first_demand_abstract.gpkg")
nodes_abs.to_file(out_abs, layer="nodes", driver="GPKG")
edges_abs.to_file(out_abs, layer="edges", driver="GPKG")

# CALCULATE STATS
d_len, d_flow = process_edges(edges_abs, reference_flow)
plot_data["demand"]["length"] = d_len
plot_data["demand"]["flow"] = d_flow
print(f"  > Demand Flow: {d_flow:.0f}, Length: {d_len:.0f}")





input_folder = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\newcastle\current_ltn_scenario"
pattern = os.path.join(input_folder, "newcastle_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle")
files = sorted(glob.glob(pattern))
print(f"Found {len(files)} random run files.")

for fpath in files:
    fname = os.path.basename(fpath)
    run_id = fname.split("_run")[-1].split(".")[0]
    print(f"Processing Random Run {run_id}...")
    with open(fpath, "rb") as f:
        data = pickle.load(f)
    G_abs = data["GT_abstracts"][1]
    if "crs" not in G_abs.graph: G_abs.graph["crs"] = "EPSG:3857"
    nodes_abs, edges_abs = ox.graph_to_gdfs(G_abs)
    
    # save for inspection
    out_abs = os.path.join(output_folder, f"random_run{run_id}_abstract.gpkg")
    nodes_abs.to_file(out_abs, layer="nodes", driver="GPKG")
    edges_abs.to_file(out_abs, layer="edges", driver="GPKG")
    
    # calcaulate stats
    r_len, r_flow = process_edges(edges_abs, reference_flow)
    plot_data["random"]["length"].append(r_len)
    plot_data["random"]["flow"].append(r_flow)

# plot
plt.figure(figsize=(10, 6))
plt.scatter(plot_data["random"]["length"], plot_data["random"]["flow"], 
            color='blue', alpha=0.6, label='Random Runs')
plt.scatter(plot_data["demand"]["length"], plot_data["demand"]["flow"], 
            color='red', s=100, marker='X', label='Demand Weighted', zorder=5)
plt.xlabel('Total Edge Length (m)')
plt.ylabel('Total Flow Sum')
plt.title('Demand Met: Demand vs Random Growth in iteration 1')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

- we can do the same but this time for the 2nd iteration just to check:

In [ ]:
# # Path to the reference demand GPKG (contains 'total_flow' per edge)
# ref_gpkg_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\newcastle\current_ltn_scenario\newcastle_current_ltn_scenario_greedy_demand_weighted.gpkg"
# print("Loading reference demand data...")
# mydemand = gpd.read_file(ref_gpkg_path)


# mydemand["edge_id"] = mydemand.apply(
#     lambda r: tuple(sorted((r.start_osmid, r.end_osmid))), axis=1)

# reference_flow = mydemand[["edge_id", "total_flow"]]
# output_folder = r"H:\Other"

# plot_data = {
#     "demand": {"flow": 0, "length": 0},
#     "random": {"flow": [], "length": []}}

# def process_edges(edges_gdf, ref_df):
#     """Helper to merge flow and calc stats"""
#     edges_gdf = edges_gdf.copy()
#     edges_gdf = edges_gdf.reset_index()
#     edges_gdf["edge_id"] = edges_gdf.apply(
#         lambda r: tuple(sorted((r.u, r.v))), axis=1)
#     merged = edges_gdf.merge(ref_df, on="edge_id", how="left")
#     total_len = merged.length.sum()
#     total_flow = merged["total_flow"].sum()
#     return total_len, total_flow


# demand_pickle_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\newcastle\current_ltn_scenario\newcastle_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle"
# print(f"Processing DEMAND scenario...")
# with open(demand_pickle_path, "rb") as f:
#     data = pickle.load(f)
# G_abs = data["GT_abstracts"][2]
# if "crs" not in G_abs.graph: G_abs.graph["crs"] = "EPSG:3857"
# nodes_abs, edges_abs = ox.graph_to_gdfs(G_abs)

# # SAVE 
# out_abs = os.path.join(output_folder, "first_demand_abstract.gpkg")
# nodes_abs.to_file(out_abs, layer="nodes", driver="GPKG")
# edges_abs.to_file(out_abs, layer="edges", driver="GPKG")

# # CALCULATE STATS
# d_len, d_flow = process_edges(edges_abs, reference_flow)
# plot_data["demand"]["length"] = d_len
# plot_data["demand"]["flow"] = d_flow
# print(f"  > Demand Flow: {d_flow:.0f}, Length: {d_len:.0f}")

# input_folder = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\newcastle\current_ltn_scenario"
# pattern = os.path.join(input_folder, "newcastle_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle")
# files = sorted(glob.glob(pattern))

# print(f"Found {len(files)} random run files.")
# for fpath in files:
#     fname = os.path.basename(fpath)
#     run_id = fname.split("_run")[-1].split(".")[0]
#     print(f"Processing Random Run {run_id}...")
#     with open(fpath, "rb") as f:
#         data = pickle.load(f)
#     G_abs = data["GT_abstracts"][2]
#     if "crs" not in G_abs.graph: G_abs.graph["crs"] = "EPSG:3857"
#     nodes_abs, edges_abs = ox.graph_to_gdfs(G_abs)
#     out_abs = os.path.join(output_folder, f"random_run{run_id}_abstract.gpkg")
#     nodes_abs.to_file(out_abs, layer="nodes", driver="GPKG")
#     edges_abs.to_file(out_abs, layer="edges", driver="GPKG")
#     r_len, r_flow = process_edges(edges_abs, reference_flow)
#     plot_data["random"]["length"].append(r_len)
#     plot_data["random"]["flow"].append(r_flow)


# plt.figure(figsize=(10, 6))
# plt.scatter(plot_data["random"]["length"], plot_data["random"]["flow"], 
#             color='blue', alpha=0.6, label='Random Runs')

# plt.scatter(plot_data["demand"]["length"], plot_data["demand"]["flow"], 
#             color='red', s=100, marker='X', label='Demand Weighted', zorder=5)

# plt.xlabel('Total Edge Length (m)')
# plt.ylabel('Total Flow Sum')
# plt.title('Demand Met: Demand vs Random Growth on iteration 2')
# plt.legend()
# plt.grid(True, linestyle='--', alpha=0.5)
# plt.savefig(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\demand_vs_random_iteration_2.png", dpi=300, bbox_inches='tight')
# plt.show()

- since those look good, we can start to plot more runs:

In [ ]:
#  Load Reference Demand
ref_gpkg_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\newcastle\current_ltn_scenario\newcastle_current_ltn_scenario_greedy_demand_weighted.gpkg"
print("Loading reference demand data...")
mydemand = gpd.read_file(ref_gpkg_path)

# Create the sorted edge_id tuple
mydemand["edge_id"] = mydemand.apply(lambda r: tuple(sorted((r.start_osmid, r.end_osmid))), axis=1)

trips_dict = dict(zip(mydemand["edge_id"], mydemand["total_flow"]))

NUM_STEPS = 100

plot_data = {
    "demand": [],        
    "random_runs": []    }


def get_demand_met_fast(G, trips_dict):
    if not G or not hasattr(G, 'edges'):
        return 0.0
    # 1. Extract all edges from the MultiDiGraph
    # 2. Sort the node pairs (min, max) so direction doesn't matter
    # 3. Use set() to drop duplicates (shouldn't be any anyway?)
    unique_edges = set(tuple(sorted((u, v))) for u, v in G.edges())
    
    return sum(trips_dict.get(edge, 0.0) for edge in unique_edges)

base_results_folder = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\newcastle\current_ltn_scenario"

results_files = {
    "betweenness_growth": os.path.join(base_results_folder, "newcastle_poi_LTNs_tessellation_betweenness_weighted_current_ltn_scenario.pickle"),
    "demand_ltn_priority": os.path.join(base_results_folder, "newcastle_poi_LTNs_tessellation_demand_ltn_priority_weighted_current_ltn_scenario.pickle"),
    "betweenness_ltn_priority": os.path.join(base_results_folder, "newcastle_poi_LTNs_tessellation_betweenness_ltn_priority_weighted_current_ltn_scenario.pickle"),}

def load_scenario_flows(pickle_path, trips_dict, steps):
    flows = [0.0] # Inject 0 for Stage 0 baseline
    with open(pickle_path, "rb") as fh:
        data = pickle.load(fh)
    graphs = data.get("GT_abstracts", [])
    for i in range(steps):
        if i >= len(graphs):
            break
        flows.append(get_demand_met_fast(graphs[i], trips_dict))
    return flows

print("Finding demand met")
results = {}
for name, path in results_files.items():
    if os.path.exists(path):
        results[name] = load_scenario_flows(path, trips_dict, NUM_STEPS)
    else:
        results[name] = [0.0]

# Load standard Demand
demand_pickle_path = os.path.join(base_results_folder, "newcastle_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle")
if os.path.exists(demand_pickle_path):
    plot_data["demand"].append(0.0) # Stage 0
    with open(demand_pickle_path, "rb") as fh:
        ddata = pickle.load(fh)
    dgraphs = ddata.get("GT_abstracts", [])
    for i in range(NUM_STEPS):
        if i >= len(dgraphs): break
        plot_data["demand"].append(get_demand_met_fast(dgraphs[i], trips_dict))

# Load Random Runs
random_pattern = os.path.join(base_results_folder, "newcastle_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle")
random_files = sorted(glob.glob(random_pattern))
for fpath in random_files: 
    with open(fpath, "rb") as fh:
        rdata = pickle.load(fh)
    rgraphs = rdata.get("GT_abstracts", [])
    run_flows = [0.0] # Stage 0
    for i in range(NUM_STEPS):
        if i >= len(rgraphs): break
        run_flows.append(get_demand_met_fast(rgraphs[i], trips_dict))
    plot_data["random_runs"].append(run_flows)

# Padding arrays to max length for plotting
max_len = max(
    len(plot_data["demand"]),
    max((len(r) for r in plot_data["random_runs"]), default=0),
    max((len(s) for s in results.values()), default=0)
)

def pad_to_len(seq, length, fill=np.nan):
    return list(seq) + [fill] * (length - len(seq))

demand_padded = pad_to_len(plot_data["demand"], max_len, fill=np.nan)
runs_padded = [pad_to_len(r, max_len, fill=np.nan) for r in plot_data["random_runs"]]
scenarios_padded = {k: pad_to_len(v, max_len, fill=np.nan) for k, v in results.items()}

arr = np.array(runs_padded, dtype=float) if runs_padded else np.empty((0, max_len))
random_mean = np.nanmean(arr, axis=0) if arr.size else np.full(max_len, np.nan)

# plot
print("Plotting results...")
plt.figure(figsize=(12, 7))
x_axis = np.arange(0, max_len) # Starts at 0 

# Random runs
for run_values in runs_padded:
    ys = np.array(run_values, dtype=float)
    mask = ~np.isnan(ys)
    if mask.sum() >= 2:
        plt.plot(x_axis[mask], ys[mask], color='blue', alpha=0.12, linewidth=0.8)

# Random Mean
if not np.all(np.isnan(random_mean)):
    plt.plot(x_axis, random_mean, color='blue', linestyle='--', alpha=0.95, linewidth=2.0, label='Random mean')

# Demand
dys = np.array(demand_padded, dtype=float)
mask_d = ~np.isnan(dys)
if mask_d.sum() >= 2:
    plt.plot(x_axis[mask_d], dys[mask_d], color='red', linestyle='--', linewidth=2.5, label='Demand weighted')

# Betweenness Growth
bg = np.array(scenarios_padded.get("betweenness_growth", []), dtype=float)
mask_bg = ~np.isnan(bg)
if mask_bg.sum() >= 2:
    plt.plot(x_axis[mask_bg], bg[mask_bg], color='orange', linestyle='-', linewidth=2.0, label='Betweenness growth')

# Demand LTN Priority
dlp = np.array(scenarios_padded.get("demand_ltn_priority", []), dtype=float)
mask_dlp = ~np.isnan(dlp)
if mask_dlp.sum() >= 2:
    plt.plot(x_axis[mask_dlp], dlp[mask_dlp], color='green', linestyle=':', linewidth=2.0, label='Demand LTN priority')

# Betweenness LTN Priority
blp = np.array(scenarios_padded.get("betweenness_ltn_priority", []), dtype=float)
mask_blp = ~np.isnan(blp)
if mask_blp.sum() >= 2:
    plt.plot(x_axis[mask_blp], blp[mask_blp], color='purple', linestyle='-', linewidth=2.0, label='Betweenness LTN priority')

plt.xlabel('Growth Step')
plt.ylabel('Cycling demand met (number of trips which can definitely be served by the current network)')
plt.title('Demand Met')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\demand_met_over_growth.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# this should match the max value in the plot
mydemand['total_flow'].sum()

this looks good at the moment - we know that we are growing our networks correctly. 

# Checking through manually

This bit of code will make plots to see actually what is happening "on the ground" for each run, to work out why random results might do so well when we measure bikeablity on a network bigger than just the abstract connections.

In [ ]:
# load the OD demand GPKG
od_gpkg_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_current_ltn_scenario_greedy_demand_weighted.gpkg"
od_demand = gpd.read_file(od_gpkg_path)



In [ ]:
# load the base bikeable network
G_bikeable_edges = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_biketrackcarall.gpkg", layer="edges")
G_bikeable_nodes = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_biketrackcarall.gpkg", layer="nodes")
G_bikeable_edges = G_bikeable_edges.set_index(["u", "v", "key"])
G_bikeable_nodes = G_bikeable_nodes.set_index("osmid")
exit_points = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\exports_gpkg\gateshead\current_ltn_scenario\gateshead_exit_points.gpkg")
G_bikeable_nx = ox.graph_from_gdfs(G_bikeable_nodes, G_bikeable_edges)

In [ ]:
# load the LTNs (for visual help)
ltns = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\scored_neighbourhoods_gateshead.gpkg")

In [ ]:
# have a look at our demand across the city
fig, ax = plt.subplots(figsize=(10, 10))
G_bikeable_edges.plot(ax=ax, color='lightgray', linewidth=0.5)
od_demand.plot(ax=ax, column='total_flow', legend=True, scheme="natural_breaks", markersize=50, alpha=0.7)
ltns.plot(ax=ax, alpha=0.5, color="orange")
plt.show()

In [ ]:
# load some results
results_pickle_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle"
with open(results_pickle_path, "rb") as f:
    results_data = pickle.load(f)

In [ ]:
# Make sure edge_id exists as a sorted tuple
od_demand["edge_id"] = od_demand.apply(lambda r: tuple(sorted((r.start_osmid, r.end_osmid))), axis=1)
trips_dict = dict(zip(od_demand["edge_id"], od_demand["total_flow"]))


def get_demand_met_fast(G, trips_dict): # same as above
    if not G or not hasattr(G, 'edges'):
        return 0.0
    unique_edges = set(tuple(sorted((u, v))) for u, v in G.edges())
    return sum(trips_dict.get(edge, 0.0) for edge in unique_edges)


target_crs = "EPSG:4326"
od_demand = od_demand.to_crs(target_crs)
G_bikeable_edges = G_bikeable_edges.to_crs(target_crs)
ltns = ltns.to_crs(target_crs)
target_demand_total = od_demand['total_flow'].sum()



for i, (G_routed, G_abs) in enumerate(zip(results_data["GTs"], results_data["GT_abstracts"])):
    current_met_flow = get_demand_met_fast(G_abs, trips_dict)

    if "crs" not in G_routed.graph:
        G_routed.graph["crs"] = "EPSG:3857"
    if "crs" not in G_abs.graph:
        G_abs.graph["crs"] = "EPSG:3857"
        
    _, edges_routed = ox.graph_to_gdfs(G_routed)
    _, edges_abs = ox.graph_to_gdfs(G_abs)
    
    edges_routed = edges_routed.to_crs(target_crs)
    edges_abs = edges_abs.to_crs(target_crs)
    
    fig, ax = plt.subplots(figsize=(10, 10))
    G_bikeable_edges.plot(ax=ax, color='lightgray', linewidth=0.5)
    od_demand.plot(ax=ax, column='total_flow', legend=False, scheme="natural_breaks", markersize=50, alpha=0.7)

    # Add Labels 
    for _, row in od_demand.iterrows():
        if row.geometry is not None:
            point = row.geometry.interpolate(0.5, normalized=True)
            ax.text(
                point.x, point.y,
                f"{row['total_flow']:.1f}",
                fontsize=5,
                ha='center',
                va='center',
                bbox=dict(facecolor='white', alpha=0.1, edgecolor='none', pad=1),
                clip_on=True
            )

    edges_routed.plot(ax=ax, color='blue', linewidth=1)
    edges_abs.plot(ax=ax, color='red', linewidth=2)
    ltns.plot(ax=ax, alpha=0.5, color="orange")

    # Zoom to AOI
    minx, miny, maxx, maxy = edges_routed.total_bounds
    pad_x = (maxx - minx) * 0.30
    pad_y = (maxy - miny) * 0.30

    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_autoscale_on(False)
    ax.set_axis_off()


    plt.title(f"Iteration {i} | Total Flow Met: {current_met_flow:.1f} of a target {target_demand_total:.1f} ({(current_met_flow/target_demand_total*100):.1f}%)")
    save_path = rf"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\gateshead_step_{i}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Saved Frame {i} | Flow Met: {current_met_flow:.1f}")

print("All frames rendered successfully!")

we can also plot the random results. This is set up to only make a plot if the demand met is greater than the demand met at the same point from the "demand" ordered data, so we can see what is going on at those points. Hopefully we get no plots...

In [ ]:
if "edge_id" not in od_demand.columns:
    od_demand["edge_id"] = od_demand.apply(lambda r: tuple(sorted((r.start_osmid, r.end_osmid))), axis=1)
trips_dict = dict(zip(od_demand["edge_id"], od_demand["total_flow"]))

def get_demand_met_fast(G, trips_dict):
    if not G or not hasattr(G, 'edges'):
        return 0.0
    unique_edges = set(tuple(sorted((u, v))) for u, v in G.edges())
    return sum(trips_dict.get(edge, 0.0) for edge in unique_edges)


target_crs = "EPSG:4326"
od_demand = od_demand.to_crs(target_crs)
G_bikeable_edges = G_bikeable_edges.to_crs(target_crs)
ltns = ltns.to_crs(target_crs)


print("Calculating Main Demand flows...")
main_met_flows = [] 

for G_abs in results_data["GT_abstracts"]:
    current_met_flow = get_demand_met_fast(G_abs, trips_dict)
    main_met_flows.append(current_met_flow)


# find any random runs that beat the main demand flow at any iteration, and plot those ones for inspection
print("Scanning Random Runs for winners...")
random_pattern = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle"
random_files = sorted(glob.glob(random_pattern))

for run_idx, fpath in enumerate(random_files):
    with open(fpath, "rb") as fh:
        rdata = pickle.load(fh)
    rgraphs = rdata.get("GT_abstracts", [])
    
    for i, G_abs in enumerate(rgraphs):
        # Ensure we don't exceed main results length
        if i >= len(main_met_flows):
            print(f"Skipping random iteration {i} - exceeds main results length.")
            continue
        random_met_flow = get_demand_met_fast(G_abs, trips_dict)
        main_met_flow = main_met_flows[i]
        
        # DID RANDOM BEAT MAIN? 
        if random_met_flow > main_met_flow:
            print(f"Run {run_idx}, Iteration {i}: Random ({random_met_flow:.1f}) beat Main ({main_met_flow:.1f})! Plotting...")
            if "crs" not in G_abs.graph:
                G_abs.graph["crs"] = "EPSG:3857"
            
            _, edges_abs = ox.graph_to_gdfs(G_abs)
            edges_abs = edges_abs.to_crs(target_crs)
            
            fig, ax = plt.subplots(figsize=(10, 10))
            G_bikeable_edges.plot(ax=ax, color='lightgray', linewidth=0.5)
            od_demand.plot(ax=ax, column='total_flow', legend=False, scheme="natural_breaks", markersize=50, alpha=0.7)

            # Add Labels 
            for _, row in od_demand.iterrows():
                if row.geometry is not None:
                    point = row.geometry.interpolate(0.5, normalized=True)
                    ax.text(
                        point.x, point.y,
                        f"{row['total_flow']:.1f}",
                        fontsize=5,
                        ha='center',
                        va='center',
                        bbox=dict(facecolor='white', alpha=0.1, edgecolor='none', pad=1),
                        clip_on=True
                    )

            edges_abs.plot(ax=ax, color='red', linewidth=2)
            ltns.plot(ax=ax, alpha=0.5, color="orange")

            # Zoom to AOI
            minx, miny, maxx, maxy = edges_abs.total_bounds
            pad_x = (maxx - minx) * 0.30
            pad_y = (maxy - miny) * 0.30

            ax.set_xlim(minx - pad_x, maxx + pad_x)
            ax.set_ylim(miny - pad_y, maxy + pad_y)
            ax.set_autoscale_on(False)
            ax.set_axis_off()

            plt.title(f"Random Run {run_idx} - Iteration {i} \n Met: {random_met_flow:.1f} vs Main: {main_met_flow:.1f}")
            
            save_path = rf"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\gateshead_random_run{run_idx}_step_{i}.png"
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig)

print("\nFinished checking all Random runs.")

----------------------------

# LTS approach

The previous checks all just used the joining of a link between two points as the assessment of if demand is met. Some places might already have demand met, and some connections might add other routes to new connections etc, so here we try a LTS approach, which reflects the "on the ground" position better.

In [ ]:
## set up G_base
G_base = G_bikeable_nx.copy()
_, edges_gdf = ox.graph_to_gdfs(G_base)
if edges_gdf.crs != ltns.crs:
    ltns = ltns.to_crs(edges_gdf.crs)
edges_gdf = edges_gdf.drop(columns=['index_left', 'index_right'], errors='ignore')
ltns = ltns.drop(columns=['index_left', 'index_right'], errors='ignore')
# This finds every edge that touches or is inside an LTN polygon
edges_in_ltn = gpd.sjoin(edges_gdf, ltns, how='inner', predicate='intersects')
ltn_edge_indices = set(edges_in_ltn.index)
ltn_flags = {}
for u, v, k in G_base.edges(keys=True):
    # If the edge index is in our joined set, flag it True
    ltn_flags[(u, v, k)] = (u, v, k) in ltn_edge_indices
nx.set_edge_attributes(G_base, ltn_flags, name='ltn_flag')
print(f"Flagged {len(ltn_edge_indices)} edges as being inside LTNs.")

lts_mapping = {
    "motorway": 4, "motorway_link": 4, "trunk": 4, "trunk_link": 4,
    "primary": 4, "primary_link": 4, "secondary": 4, "secondary_link": 4,
    "tertiary": 3, "tertiary_link": 3, "unclassified": 3,
    "residential": 2, "living_street": 2,
    "cycleway": 1, "track": 1, "path": 1, "bridleway": 1, "footway": 1, "pedestrian": 1}

# if unknown
DEFAULT_LTS = 4 

for u, v, key, data in G_base.edges(keys=True, data=True):
    is_ltn = data.get('ltn_flag')
    if is_ltn is True or str(is_ltn).lower() == 'true':
        lts_class = 1
    else:
        highway_type = data.get('highway')
        if isinstance(highway_type, list):
            highway_type = highway_type
        
        # Look up the LTS class, using the default if the type isn't found
        lts_class = lts_mapping.get(highway_type, DEFAULT_LTS)
    
    # Get the physical length
    edge_length = data.get('length', 0)
    
    # Assign the new attributes to the edge
    data['lts_class'] = lts_class
    data['lts_length'] = edge_length * lts_class

if G_base.is_directed():
    G_base.to_undirected()


- first approach is to get rid of any streets with a LTS greater than 2. This is the "binary" option for LTS bikeable trips calculation

In [ ]:
G_LTS_2 = G_base.copy()
edges_to_remove = []
for u, v, key, data in G_LTS_2.edges(keys=True, data=True):
    if data.get('lts_class', 4) > 2:
        edges_to_remove.append((u, v, key))
G_LTS_2.remove_edges_from(edges_to_remove)


In [ ]:
# Extract the Start and End points
start_points = od_demand['geometry'].apply(lambda line: line.interpolate(0, normalized=True) if line is not None else None)
end_points = od_demand['geometry'].apply(lambda line: line.interpolate(1, normalized=True) if line is not None else None)
gdf_starts = gpd.GeoDataFrame(geometry=start_points, crs=od_demand.crs)
gdf_ends = gpd.GeoDataFrame(geometry=end_points, crs=od_demand.crs)
if ltns.crs != od_demand.crs:
    ltns = ltns.to_crs(od_demand.crs)
ltns_clean = ltns.drop(columns=['index_left', 'index_right'], errors='ignore').copy()
ltns_clean['neighbourhood_id'] = ltns_clean.index

# spatial joins
starts_joined = gpd.sjoin(gdf_starts, ltns_clean, how='left', predicate='intersects')
ends_joined = gpd.sjoin(gdf_ends, ltns_clean, how='left', predicate='intersects')

#  Map back to od_demand
od_demand['start_neighbourhood_id'] = starts_joined['neighbourhood_id']
od_demand['end_neighbourhood_id'] = ends_joined['neighbourhood_id']
od_demand['ltn_origin'] = od_demand['start_neighbourhood_id'].notna()
od_demand['ltn_destination'] = od_demand['end_neighbourhood_id'].notna()


In [ ]:
# Build the fast lookup dictionary for exit points
exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(list).to_dict()


processed_trips = []
for s_node, e_node, flow, is_ltn_o, is_ltn_d, s_hood, e_hood in zip(
    od_demand['start_osmid'], od_demand['end_osmid'], od_demand['total_flow'],
    od_demand['ltn_origin'], od_demand['ltn_destination'],
    od_demand['start_neighbourhood_id'], od_demand['end_neighbourhood_id']):
    if pd.isna(s_node) or pd.isna(e_node):
        continue  
        
    start_targets = [int(s_node)]
    end_targets = [int(e_node)]
    
    if is_ltn_o and pd.notna(s_hood) and int(s_hood) in exit_dict:
        start_targets = exit_dict[int(s_hood)]
            
    if is_ltn_d and pd.notna(e_hood) and int(e_hood) in exit_dict:
        end_targets = exit_dict[int(e_hood)]
        
    processed_trips.append({
        'starts': start_targets,
        'ends': end_targets,
        'flow': flow if pd.notna(flow) else 0})

# prep base graph
G_base_undir = G_LTS_2.to_undirected()
bikeable_flows_per_stage = []

print(f"Routing {len(processed_trips)} trips across {len(results_data['GTs'])} stages...")
for i, G_routed in enumerate(results_data["GTs"]):
    G_current = G_base_undir.copy()
    G_current.add_edges_from(G_routed.edges()) # Add the new built infrastructure
    
    # Map every node to an 'neighbourhood ID' instantly. 
    node_to_neighbourhood = {}
    for neighbourhood_id, component_nodes in enumerate(nx.connected_components(G_current)):
        for node in component_nodes:
            node_to_neighbourhood[node] = neighbourhood_id

    stage_total_flow = 0
    missing_nodes_count = 0
    no_path_count = 0
    success_count = 0

    for trip in processed_trips:
        # Look up the neighbourhood ID for every valid start and end node
        start_neighbourhoods = set(node_to_neighbourhood[s] for s in trip['starts'] if s in node_to_neighbourhood)
        end_neighbourhoods = set(node_to_neighbourhood[e] for e in trip['ends'] if e in node_to_neighbourhood)
        
        # If either set is empty, it means none of the nodes exist in the graph at all
        if not start_neighbourhoods or not end_neighbourhoods:
            missing_nodes_count += 1
            continue
    
        if start_neighbourhoods.intersection(end_neighbourhoods):
            success_count += 1
            stage_total_flow += trip['flow']
        else:
            no_path_count += 1
    bikeable_flows_per_stage.append(stage_total_flow)
    print(f"Stage {i} | Flow: {stage_total_flow:.1f} | Success: {success_count} routes | No Path: {no_path_count} | Missing Nodes: {missing_nodes_count}")

the final flow should be the same as the sum of OD_demand:

In [ ]:
od_demand['total_flow'].sum()

- second option is to have a max distance and route using the lts distance

In [ ]:
# Build the fast lookup dictionary for exit points
exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(list).to_dict()
print("Pre-computing trip targets...")
processed_trips = []

for s_node, e_node, flow, is_ltn_o, is_ltn_d, s_hood, e_hood in zip(
    od_demand['start_osmid'], od_demand['end_osmid'], od_demand['total_flow'],
    od_demand['ltn_origin'], od_demand['ltn_destination'],
    od_demand['start_neighbourhood_id'], od_demand['end_neighbourhood_id']
):
    if pd.isna(s_node) or pd.isna(e_node):
        continue  
        
    start_targets = [int(s_node)]
    end_targets = [int(e_node)]
    
    if is_ltn_o and pd.notna(s_hood) and int(s_hood) in exit_dict:
        start_targets = exit_dict[int(s_hood)]
            
    if is_ltn_d and pd.notna(e_hood) and int(e_hood) in exit_dict:
        end_targets = exit_dict[int(e_hood)]
        
    processed_trips.append({
        'starts': start_targets,
        'ends': end_targets,
        'flow': flow if pd.notna(flow) else 0
    })


# We use the FULL G_base this time, not the filtered G_LTS_2
# Make sure G_base already has 'lts_length' calculated on its edges!
G_base_undir = G_base.to_undirected() 
bikeable_flows_per_stage = []

print(f"Routing {len(processed_trips)} trips across {len(results_data['GTs'])} stages with a 5000m lts_length cutoff...")

for i, G_routed in enumerate(results_data["GTs"]):
    G_current = G_base_undir.copy()
    new_edges_formatted = []
    for u, v, k, d in G_routed.edges(keys=True, data=True):
        edge_length = d.get('length', 0)
        d['lts_class'] = 1
        d['lts_length'] = edge_length * 1 
        new_edges_formatted.append((u, v, k, d))
        
    G_current.add_edges_from(new_edges_formatted)
    
    stage_total_flow = 0
    missing_nodes_count = 0
    no_path_count = 0
    success_count = 0

    for trip in processed_trips:
        valid_starts = [s for s in trip['starts'] if G_current.has_node(s)]
        valid_ends = set(e for e in trip['ends'] if G_current.has_node(e))
        
        if not valid_starts or not valid_ends:
            missing_nodes_count += 1
            continue
            
        try:
            # This searches outward from ALL valid start points simultaneously.
            # It strictly stops searching if a path exceeds 5000 lts_length units.
            lengths = nx.multi_source_dijkstra_path_length(
                G_current, 
                valid_starts, 
                cutoff=5000, 
                weight='lts_length'
            )
            
            # Check if any of our valid end destinations were reached within the limit
            reached_ends = valid_ends.intersection(lengths.keys())
            
            if reached_ends:
                success_count += 1
                stage_total_flow += trip['flow']
            else:
                # If we searched up to 5000m and didn't hit an end node, it's a fail.
                no_path_count += 1
                
        except nx.NodeNotFound:
            missing_nodes_count += 1

    bikeable_flows_per_stage.append(stage_total_flow)
    
    print(f"Stage {i} | Flow: {stage_total_flow:.1f} | Success: {success_count} | Too Long/No Path: {no_path_count} | Missing: {missing_nodes_count}")

-------

## NO CELLS ABOVE HERE NEED TO BE RUN

I've turned those two approaches into functions: 

In [ ]:
def get_bikeablity_lts_2_only(results_gts, G_LTS_2, od_demand, exit_points):
    """
    Evaluates network bikeability using strict graph connectivity.
    Only allows paths on LTS 1 or 2 infrastructure. Extremely fast.
    """
    #  Build lookup dictionary
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(list).to_dict()

    # Pre-compute trip targets
    processed_trips = []
    for s_node, e_node, flow, is_ltn_o, is_ltn_d, s_hood, e_hood in zip(
        od_demand['start_osmid'], od_demand['end_osmid'], od_demand['total_flow'],
        od_demand['ltn_origin'], od_demand['ltn_destination'],
        od_demand['start_neighbourhood_id'], od_demand['end_neighbourhood_id'] ):
        if pd.isna(s_node) or pd.isna(e_node):
            continue  
            
        start_targets = [int(s_node)]
        end_targets = [int(e_node)]
        
        if is_ltn_o and pd.notna(s_hood) and int(s_hood) in exit_dict:
            start_targets = exit_dict[int(s_hood)]
                
        if is_ltn_d and pd.notna(e_hood) and int(e_hood) in exit_dict:
            end_targets = exit_dict[int(e_hood)]
            
        processed_trips.append({
            'starts': start_targets,
            'ends': end_targets,
            'flow': flow if pd.notna(flow) else 0
        })

    # Prepare Base Graph (Filtered LTS <= 2)
    G_base_undir = G_LTS_2.to_undirected()
    bikeable_flows_per_stage = []

    print(f"[Strict LTS <= 2] Routing {len(processed_trips)} trips across {len(results_gts)} stages...")
    
    # Routing Loop
    for i, G_routed in enumerate(results_gts):
        G_current = G_base_undir.copy()
        G_current.add_edges_from(G_routed.edges())
        
       
        node_to_neighbourhood = {}
        for island_id, component_nodes in enumerate(nx.connected_components(G_current)):
            for node in component_nodes:
                node_to_neighbourhood[node] = island_id

        stage_total_flow = 0
        missing_nodes_count = 0
        no_path_count = 0
        success_count = 0

        for trip in processed_trips:
            # Look up Island IDs
            start_neighbourhoods = set(node_to_neighbourhood[s] for s in trip['starts'] if s in node_to_neighbourhood)
            end_neighbourhoods = set(node_to_neighbourhood[e] for e in trip['ends'] if e in node_to_neighbourhood)
            
            if not start_neighbourhoods or not end_neighbourhoods:
                missing_nodes_count += 1
                continue
        
            # If the islands intersect, a path exists
            if start_neighbourhoods.intersection(end_neighbourhoods):
                success_count += 1
                stage_total_flow += trip['flow']
            else:
                no_path_count += 1
                
        bikeable_flows_per_stage.append(stage_total_flow)
        print(f"  Stage {i} | Flow: {stage_total_flow:.1f} | Success: {success_count} routes | No Path: {no_path_count} | Missing Nodes: {missing_nodes_count}")
        
    return bikeable_flows_per_stage



    
def get_bikeablity_lts_length_cutoff(results_gts, G_base, od_demand, exit_points, cutoff=5000):
    """
    Evaluates network bikeability using a stress-distance cutoff.
    Allows all LTS levels, but fails trips if the total 'lts_length' exceeds the cutoff.
    """
    # Build lookup dictionary
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(list).to_dict()
    
    # Pre-compute trip targets
    processed_trips = []
    for s_node, e_node, flow, is_ltn_o, is_ltn_d, s_hood, e_hood in zip(
        od_demand['start_osmid'], od_demand['end_osmid'], od_demand['total_flow'],
        od_demand['ltn_origin'], od_demand['ltn_destination'],
        od_demand['start_neighbourhood_id'], od_demand['end_neighbourhood_id']
    ):
        if pd.isna(s_node) or pd.isna(e_node):
            continue  
            
        start_targets = [int(s_node)]
        end_targets = [int(e_node)]
        
        if is_ltn_o and pd.notna(s_hood) and int(s_hood) in exit_dict:
            start_targets = exit_dict[int(s_hood)]
                
        if is_ltn_d and pd.notna(e_hood) and int(e_hood) in exit_dict:
            end_targets = exit_dict[int(e_hood)]
            
        processed_trips.append({
            'starts': start_targets,
            'ends': end_targets,
            'flow': flow if pd.notna(flow) else 0
        })

    
    G_base_undir = G_base.to_undirected() 
    bikeable_flows_per_stage = []

    for i, G_routed in enumerate(results_gts):
        G_current = G_base_undir.copy()
        
        # Format and add new infrastructure
        new_edges_formatted = []
        for u, v, k, d in G_routed.edges(keys=True, data=True):
            edge_length = d.get('length', 0)
            d['lts_class'] = 1
            d['lts_length'] = edge_length * 1 
            new_edges_formatted.append((u, v, k, d))
            
        G_current.add_edges_from(new_edges_formatted)
        
        stage_total_flow = 0
        missing_nodes_count = 0
        no_path_count = 0
        success_count = 0

        for trip in processed_trips:
            valid_starts = [s for s in trip['starts'] if G_current.has_node(s)]
            valid_ends = set(e for e in trip['ends'] if G_current.has_node(e))
            
            if not valid_starts or not valid_ends:
                missing_nodes_count += 1
                continue
                
            try:
                lengths = nx.multi_source_dijkstra_path_length(
                    G_current, 
                    valid_starts, 
                    cutoff=cutoff, 
                    weight='lts_length'
                )
                
                reached_ends = valid_ends.intersection(lengths.keys())
                
                if reached_ends:
                    success_count += 1
                    stage_total_flow += trip['flow']
                else:
                    no_path_count += 1
                    
            except nx.NodeNotFound:
                missing_nodes_count += 1

        bikeable_flows_per_stage.append(stage_total_flow)
        print(f"  Stage {i} | Flow: {stage_total_flow:.1f} | Success: {success_count} | Too Long/No Path: {no_path_count} | Missing: {missing_nodes_count}")
        
    return bikeable_flows_per_stage

In [ ]:
# load the OD demand GPKG
od_gpkg_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_current_ltn_scenario_greedy_demand_weighted.gpkg"
od_demand = gpd.read_file(od_gpkg_path)

# load the base bikeable network
G_bikeable_edges = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_biketrackcarall.gpkg", layer="edges")
G_bikeable_nodes = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_biketrackcarall.gpkg", layer="nodes")
G_bikeable_edges = G_bikeable_edges.set_index(["u", "v", "key"])
G_bikeable_nodes = G_bikeable_nodes.set_index("osmid")
exit_points = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\exports_gpkg\gateshead\current_ltn_scenario\gateshead_exit_points.gpkg")
G_bikeable_nx = ox.graph_from_gdfs(G_bikeable_nodes, G_bikeable_edges)

# load the LTNs (for visual help)
ltns = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\scored_neighbourhoods_gateshead.gpkg")

In [ ]:
## set up G_base
G_base = G_bikeable_nx.copy()
_, edges_gdf = ox.graph_to_gdfs(G_base)
if edges_gdf.crs != ltns.crs:
    ltns = ltns.to_crs(edges_gdf.crs)
edges_gdf = edges_gdf.drop(columns=['index_left', 'index_right'], errors='ignore')
ltns = ltns.drop(columns=['index_left', 'index_right'], errors='ignore')
# This finds every edge that touches or is inside an LTN polygon
edges_in_ltn = gpd.sjoin(edges_gdf, ltns, how='inner', predicate='intersects')
ltn_edge_indices = set(edges_in_ltn.index)
ltn_flags = {}
for u, v, k in G_base.edges(keys=True):
    # If the edge index is in our joined set, flag it True
    ltn_flags[(u, v, k)] = (u, v, k) in ltn_edge_indices
nx.set_edge_attributes(G_base, ltn_flags, name='ltn_flag')
print(f"Flagged {len(ltn_edge_indices)} edges as being inside LTNs.")

lts_mapping = {
    "motorway": 4, "motorway_link": 4, "trunk": 4, "trunk_link": 4,
    "primary": 4, "primary_link": 4, "secondary": 4, "secondary_link": 4,
    "tertiary": 3, "tertiary_link": 3, "unclassified": 3,
    "residential": 2, "living_street": 2,
    "cycleway": 1, "track": 1, "path": 1, "bridleway": 1, "footway": 1, "pedestrian": 1}

# if unknown
DEFAULT_LTS = 4 

for u, v, key, data in G_base.edges(keys=True, data=True):
    is_ltn = data.get('ltn_flag')
    if is_ltn is True or str(is_ltn).lower() == 'true':
        lts_class = 1
    else:
        highway_type = data.get('highway')
        if isinstance(highway_type, list):
            highway_type = highway_type
        
        # Look up the LTS class, using the default if the type isn't found
        lts_class = lts_mapping.get(highway_type, DEFAULT_LTS)
    
    # Get the physical length
    edge_length = data.get('length', 0)
    
    # Assign the new attributes to the edge
    data['lts_class'] = lts_class
    data['lts_length'] = edge_length * lts_class

if G_base.is_directed():
    G_base.to_undirected()

G_LTS_2 = G_base.copy()
edges_to_remove = []
for u, v, key, data in G_LTS_2.edges(keys=True, data=True):
    if data.get('lts_class', 4) > 2:
        edges_to_remove.append((u, v, key))
G_LTS_2.remove_edges_from(edges_to_remove)


In [ ]:
# load some more results
demand = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle"
with open(demand, "rb") as f:
    demand_data = pickle.load(f)
    demand_gts = demand_data.get("GTs", [])
    demand_abs = demand_data.get("GT_abstracts", [])
    demand_gts.insert(0, nx.MultiGraph()) # add an empty one
    demand_abs.insert(0, nx.MultiGraph())
print("Demand Loaded")

betweeness = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_betweenness_weighted_current_ltn_scenario.pickle"
with open(betweeness, "rb") as f:
    betweenness_data = pickle.load(f)
    betweenness_gts = betweenness_data.get("GTs", [])
    betweenness_abs = betweenness_data.get("GT_abstracts", [])
    betweenness_gts.insert(0, nx.MultiGraph())
    betweenness_abs.insert(0, nx.MultiGraph())
print("Betweeness Loaded")

betweeness_ltn_priority = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_betweenness_ltn_priority_weighted_current_ltn_scenario.pickle"
with open(betweeness_ltn_priority, "rb") as f:
    betweenness_ltn_priority_data = pickle.load(f)
    betweenness_ltn_priority_gts = betweenness_ltn_priority_data.get("GTs", [])
    betweenness_ltn_priority_abs = betweenness_ltn_priority_data.get("GT_abstracts", [])
    betweenness_ltn_priority_gts.insert(0, nx.MultiGraph())
    betweenness_ltn_priority_abs.insert(0, nx.MultiGraph())
print("Betweeness LTN Loaded")

demand_ltn_priority = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_demand_ltn_priority_weighted_current_ltn_scenario.pickle"
with open(demand_ltn_priority, "rb") as f:
    demand_ltn_priority_data = pickle.load(f)
    demand_ltn_priority_gts = demand_ltn_priority_data.get("GTs", [])
    demand_ltn_priority_abs = demand_ltn_priority_data.get("GT_abstracts", [])
    demand_ltn_priority_gts.insert(0, nx.MultiGraph())
    demand_ltn_priority_abs.insert(0, nx.MultiGraph())
print("Demand LTN Loaded")

random_runs = glob.glob(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle")
random_gts_list = []
random_abs_list = []
for run in random_runs[:25]: # just 25 for speed
    with open(run, "rb") as f:
        random_data = pickle.load(f)
        r_gts = random_data.get("GTs", [])
        r_abs = random_data.get("GT_abstracts", [])
        r_gts.insert(0, nx.MultiGraph())
        r_abs.insert(0, nx.MultiGraph())
        random_gts_list.append(r_gts)
        random_abs_list.append(r_abs)
print("Random Loaded")

# trim to just 10 random for speed
random_gts_list = random_gts_list[:25]
random_abs_list = random_abs_list[:25]



In [ ]:
# Extract the Start and End points
start_points = od_demand['geometry'].apply(lambda line: line.interpolate(0, normalized=True) if line is not None else None)
end_points = od_demand['geometry'].apply(lambda line: line.interpolate(1, normalized=True) if line is not None else None)
gdf_starts = gpd.GeoDataFrame(geometry=start_points, crs=od_demand.crs)
gdf_ends = gpd.GeoDataFrame(geometry=end_points, crs=od_demand.crs)
if ltns.crs != od_demand.crs:
    ltns = ltns.to_crs(od_demand.crs)
ltns_clean = ltns.drop(columns=['index_left', 'index_right'], errors='ignore').copy()
ltns_clean['neighbourhood_id'] = ltns_clean.index

# spatial joins
starts_joined = gpd.sjoin(gdf_starts, ltns_clean, how='left', predicate='intersects')
ends_joined = gpd.sjoin(gdf_ends, ltns_clean, how='left', predicate='intersects')

#  Map back to od_demand
od_demand['start_neighbourhood_id'] = starts_joined['neighbourhood_id']
od_demand['end_neighbourhood_id'] = ends_joined['neighbourhood_id']
od_demand['ltn_origin'] = od_demand['start_neighbourhood_id'].notna()
od_demand['ltn_destination'] = od_demand['end_neighbourhood_id'].notna()


In [ ]:
print("--- RUNNING STRICT LTS <= 2 METHOD ---")
demand_strict = get_bikeablity_lts_2_only(demand_gts, G_LTS_2, od_demand, exit_points)
betweenness_strict = get_bikeablity_lts_2_only(betweenness_gts, G_LTS_2, od_demand, exit_points)
demand_ltn_priority_strict = get_bikeablity_lts_2_only(demand_ltn_priority_gts, G_LTS_2, od_demand, exit_points)
betweeness_ltn_prioirty_strict = get_bikeablity_lts_2_only(betweenness_ltn_priority_gts, G_LTS_2, od_demand, exit_points)
random_strict_all = []
for i, r_gts in enumerate(random_gts_list):
    print(f"\nEvaluating Random Run {i+1} of {len(random_gts_list)}...")
    flows = get_bikeablity_lts_2_only(r_gts, G_LTS_2, od_demand, exit_points)
    random_strict_all.append(flows)
    
# Calculate the mean across all random runs for each stage
mean_random_strict = np.mean(random_strict_all, axis=0)

print("\n\n--- RUNNING 5000m CUTOFF METHOD ---")
demand_cutoff = get_bikeablity_lts_length_cutoff(demand_gts, G_base, od_demand, exit_points, cutoff=5000)
betweenness_cutoff = get_bikeablity_lts_length_cutoff(betweenness_gts, G_base, od_demand, exit_points, cutoff=5000)
demand_ltn_priority_cutoff = get_bikeablity_lts_length_cutoff(demand_ltn_priority_gts, G_base, od_demand, exit_points, cutoff=5000)
betweeness_ltn_prioirty_cutoff = get_bikeablity_lts_length_cutoff(betweenness_ltn_priority_gts, G_base, od_demand, exit_points, cutoff=5000)
random_cutoff_all = []
for i, r_gts in enumerate(random_gts_list):
    print(f"\nEvaluating Random Run {i+1} of {len(random_gts_list)}...")
    flows = get_bikeablity_lts_length_cutoff(r_gts, G_base, od_demand, exit_points, cutoff=5000)
    random_cutoff_all.append(flows)

# Calculate the mean across all random runs for each stage
mean_random_cutoff = np.mean(random_cutoff_all, axis=0)



In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
stages = range(len(demand_strict))


# PLOT 1
for i, r_flows in enumerate(random_strict_all):
    label = 'Random (Individual)' if i == 0 else None
    ax1.plot(stages, r_flows, color='blue', alpha=0.15, linewidth=1.5, label=label)
ax1.plot(stages, mean_random_strict, color='blue', linestyle='--', linewidth=2.5, label='Random (Mean)')
ax1.plot(stages, betweenness_strict, color='orange', linestyle='-', linewidth=2.5, label='Betweenness')
ax1.plot(stages, demand_strict, color='red', linestyle='--', linewidth=2.5, label='Demand')
ax1.plot(stages, demand_ltn_priority_strict, color='green', linestyle='--', linewidth=2.5, label='Demand LTN Priority')
ax1.plot(stages, betweeness_ltn_prioirty_strict, color='purple', linestyle='--', linewidth=2.5, label='Betweenness LTN Priority')
ax1.set_title("Method 1: Strict LTS <= 2 Connectivity", fontsize=14, fontweight='bold')
ax1.set_xlabel("Infrastructure Added (Stages)", fontsize=12)
ax1.set_ylabel("Total Connected Flow", fontsize=12)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='lower right', fontsize=10)


# PLOT 2
for i, r_flows in enumerate(random_cutoff_all):
    label = 'Random (Individual)' if i == 0 else None
    ax2.plot(stages, r_flows, color='blue', alpha=0.15, linewidth=1.5, label=label)
ax2.plot(stages, mean_random_cutoff, color='blue', linestyle='--', linewidth=2.5, label='Random (Mean)')
ax2.plot(stages, betweenness_cutoff, color='orange', linestyle='-', linewidth=2.5, label='Betweenness')
ax2.plot(stages, demand_cutoff, color='red', linestyle='--', linewidth=2.5, label='Demand')
ax2.plot(stages, demand_ltn_priority_cutoff, color='green', linestyle='--', linewidth=2.5, label='Demand LTN Priority')
ax2.plot(stages, betweeness_ltn_prioirty_cutoff, color='purple', linestyle='--', linewidth=2.5, label='Betweenness LTN Priority')
ax2.set_title("Method 2: 5000m LTS Length Cutoff", fontsize=14, fontweight='bold')
ax2.set_xlabel("Infrastructure Added (Stages)", fontsize=12)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\gateshead_bikeability_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
G_bikeable_edges.columns

In [ ]:
# set up folder
base_debug_dir = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging"

# Create specific folders so things stay organized
folders = {
    "strict_main": os.path.join(base_debug_dir, "Strict_LTS", "Main_Scenarios"),
    "strict_random": os.path.join(base_debug_dir, "Strict_LTS", "Random_Winners"),
}

for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

# Ensure base layers are ready
target_crs = "EPSG:4326"
od_demand = od_demand.to_crs(target_crs)
G_bikeable_edges = G_bikeable_edges.to_crs(target_crs) 
ltns = ltns.to_crs(target_crs)
target_demand_total = od_demand['total_flow'].sum()


def generate_map(G_routed, G_abs, met_flow, stage, title, save_path):
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Base Map & Demand
    G_bikeable_edges.plot(ax=ax, color='lightgray', linewidth=0.5)
    od_demand.plot(ax=ax, column='total_flow', legend=False, scheme="natural_breaks", markersize=50, alpha=0.7)

    # Add Labels 
    for _, row in od_demand.iterrows():
        if row.geometry is not None:
            point = row.geometry.interpolate(0.5, normalized=True)
            ax.text(
                point.x, point.y,
                f"{row['total_flow']:.1f}",
                fontsize=5, ha='center', va='center',
                bbox=dict(facecolor='white', alpha=0.1, edgecolor='none', pad=1),
                clip_on=True
            )

    # Process and Plot Routed Edges (Blue)
    # Check if edges exist to prevent Stage 0 crash
    has_routed = G_routed is not None and len(G_routed.edges) > 0
    if has_routed:
        if "crs" not in G_routed.graph:
            G_routed.graph["crs"] = "EPSG:3857"
        _, edges_routed = ox.graph_to_gdfs(G_routed)
        edges_routed = edges_routed.to_crs(target_crs)
        edges_routed.plot(ax=ax, color='blue', linewidth=1)
        
    # Process and Plot Abstract Edges (Red)
    has_abs = G_abs is not None and len(G_abs.edges) > 0
    if has_abs:
        if "crs" not in G_abs.graph:
            G_abs.graph["crs"] = "EPSG:3857"
        _, edges_abs = ox.graph_to_gdfs(G_abs)
        edges_abs = edges_abs.to_crs(target_crs)
        edges_abs.plot(ax=ax, color='red', linewidth=2)

    # Plot LTNs
    ltns.plot(ax=ax, alpha=0.5, color="orange")

    # Zoom to AOI
    if has_routed:
        minx, miny, maxx, maxy = edges_routed.total_bounds
    else:
        # If Stage 0, zoom to the overall demand area instead
        minx, miny, maxx, maxy = od_demand.total_bounds
        
    pad_x = (maxx - minx) * 0.30
    pad_y = (maxy - miny) * 0.30

    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_autoscale_on(False)
    ax.set_axis_off()

    # Title and Save
    pct_met = (met_flow / target_demand_total) * 100 if target_demand_total > 0 else 0
    plt.title(f"{title} | Stage {stage}\nFlow Met: {met_flow:.1f} of {target_demand_total:.1f} ({pct_met:.1f}%)")
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


for stage, (G_routed, G_abs, flow) in enumerate(zip(demand_gts, demand_abs, demand_strict)):
    save_path = os.path.join(folders["strict_main"], f"Demand_Stage_{stage}.png")
    generate_map(G_routed, G_abs, flow, stage, "Demand Weighted", save_path)


for run_idx, (r_gts, r_abs, r_flows) in enumerate(zip(random_gts_list, random_abs_list, random_strict_all)):
    for stage, (G_routed, G_abs, random_flow) in enumerate(zip(r_gts, r_abs, r_flows)):
        
        # Guard against index errors
        if stage >= len(demand_strict):
            continue
            
        main_flow = demand_strict[stage]
        
        # If Random beats the main Demand run
        if random_flow > main_flow:
            print(f"Random Run {run_idx} Stage {stage} ({random_flow:.1f} vs {main_flow:.1f})")
            
            title = f"Random Run {run_idx} vs Demand ({random_flow:.1f} vs {main_flow:.1f})"
            save_path = os.path.join(folders["strict_random"], f"Run{run_idx}_Stage{stage}_Winner.png")
            
            generate_map(G_routed, G_abs, random_flow, stage, title, save_path)



In [ ]:
print("Setting up folders for distance Cutoff Method...")
cutoff_folders = {
    "cutoff_main": os.path.join(base_debug_dir, "Cutoff_5000m", "Main_Scenarios"),
    "cutoff_random": os.path.join(base_debug_dir, "Cutoff_5000m", "Random_Winners"),
}

for folder in cutoff_folders.values():
    os.makedirs(folder, exist_ok=True)


for stage, (G_routed, G_abs, flow) in enumerate(zip(demand_gts, demand_abs, demand_cutoff)):
    save_path = os.path.join(cutoff_folders["cutoff_main"], f"Demand_Cutoff_Stage_{stage}.png")
    generate_map(G_routed, G_abs, flow, stage, "Demand (5000m Cutoff)", save_path)


for run_idx, (r_gts, r_abs, r_flows) in enumerate(zip(random_gts_list, random_abs_list, random_cutoff_all)):
    for stage, (G_routed, G_abs, random_flow) in enumerate(zip(r_gts, r_abs, r_flows)):
        
        # Guard against index errors
        if stage >= len(demand_cutoff):
            continue
            
        main_flow = demand_cutoff[stage]
        
        # If Random beats the main Demand run in the Cutoff method
        if random_flow > main_flow:
            print(f" Random Run {run_idx} Stage {stage} ({random_flow:.1f} vs {main_flow:.1f})")
            
            title = f"Random {run_idx} vs Demand Cutoff\n({random_flow:.1f} vs {main_flow:.1f})"
            save_path = os.path.join(cutoff_folders["cutoff_random"], f"Run{run_idx}_Stage{stage}_Cutoff_Winner.png")
            
            # Use the exact same generate_map function from the previous cell!
            generate_map(G_routed, G_abs, random_flow, stage, title, save_path)



--------------

Trying to solve via https://github.com/Froguin99/ltn-bikenetwork-growth/issues/49#issuecomment-4090849075

user benefit = cost savings × demand

or in our case, bikeablity = lts distance saved (lts distance without infrastucture - lts distance with) x demand (the trips from demand) 

In [ ]:
import igraph as ig


def get_bikeability_igraph(results_gts, G_base, od_demand, exit_points, penalty_cost=20000):
    # convert to base graph
    g_base_ig = ig.Graph.from_networkx(G_base.to_undirected())
    
    # Create a mapping from OSMID to igraph index for look ups
    osmid_to_idx = {osmid: idx for idx, osmid in enumerate(g_base_ig.vs['_nx_name'])}
    
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(
        lambda x: [osmid_to_idx[o] for o in x if o in osmid_to_idx]
    ).to_dict()

    # Prepare Trips: Convert all OSMIDs to igraph internal indices
    trips = []
    total_demand = 0.0
    for _, row in od_demand.iterrows():
        if pd.isna(row['start_osmid']) or row['total_flow'] <= 0:
            continue
            
        s_idxs = exit_dict.get(int(row['start_neighbourhood_id']), []) if row['ltn_origin'] else []
        if not s_idxs and row['start_osmid'] in osmid_to_idx:
            s_idxs = [osmid_to_idx[row['start_osmid']]]
            
        e_idxs = exit_dict.get(int(row['end_neighbourhood_id']), []) if row['ltn_destination'] else []
        if not e_idxs and row['end_osmid'] in osmid_to_idx:
            e_idxs = [osmid_to_idx[row['end_osmid']]]
            
        if s_idxs and e_idxs:
            trips.append({'starts': s_idxs, 'ends': e_idxs, 'flow': row['total_flow']})
            total_demand += row['total_flow']

    if total_demand == 0: return [0.0] * len(results_gts)

    # Function to compute stage costs using igraph's distance method
    def compute_stage_costs(graph):
        costs = []
        for t in trips:
            # distances(source, targets, weights) returns a list of lists
            dists = graph.distances(source=t['starts'], target=t['ends'], weights='lts_length')
            # Get the minimum distance found across all start/end combinations
            min_dist = np.min(dists)
            costs.append(min_dist if min_dist < penalty_cost else penalty_cost)
        return np.array(costs)

    print(f"Calculating Baseline for {len(trips)} trips...")
    base_costs = compute_stage_costs(g_base_ig)
    stage_results = []

    for i, G_routed in enumerate(results_gts):
        # Work on a copy of the base igraph
        g_current = g_base_ig.copy()
        
        if G_routed is not None and len(G_routed.edges) > 0:
            new_edges = []
            new_weights = []
            for u, v, k, d in G_routed.edges(keys=True, data=True):
                if u in osmid_to_idx and v in osmid_to_idx:
                    new_edges.append((osmid_to_idx[u], osmid_to_idx[v]))
                    new_weights.append(d.get('length', 0))
            
            g_current.add_edges(new_edges)
            start_idx = g_base_ig.ecount()
            for idx, w in enumerate(new_weights):
                g_current.es[start_idx + idx]['lts_length'] = w

        current_costs = compute_stage_costs(g_current)
        savings = np.maximum(0, base_costs - current_costs)
        avg_savings = np.sum(savings * [t['flow'] for t in trips]) / total_demand
        stage_results.append(avg_savings)
        print(f"  [{run_id}] Stage {i} | Avg Savings: {avg_savings:.1f} stress-meters per cyclist")

    return stage_results

In [ ]:
from joblib import Parallel, delayed

# for speed's sake we'll sample while testing
demand_gts_sampled = demand_gts[::10]
demand_ltn_priority_gts_sampled = demand_ltn_priority_gts[::10]
betweenness_ltn_priority_gts_sampled = betweenness_ltn_priority_gts[::10]
betweenness_gts_sampled = betweenness_gts[::10]
random_gts_list_sampled = [r_gts[::10] for r_gts in random_gts_list[:10]]


demand_continuous_scores = get_bikeability_igraph(
    demand_gts_sampled, G_base, od_demand, exit_points)
demand_ltn_priority_continuous_scores = get_bikeability_igraph(
    demand_ltn_priority_gts_sampled, G_base, od_demand, exit_points)
betweenness_ltn_priority_continuous_scores = get_bikeability_igraph(
    betweenness_ltn_priority_gts_sampled, G_base, od_demand, exit_points)
betweenness_continuous_scores = get_bikeability_igraph(
    betweenness_gts_sampled, G_base, od_demand, exit_points)


print(f"\nEvaluating {len(random_gts_list_sampled)} Random Runs in parallel...")
random_continuous_all = Parallel(n_jobs=-2, verbose=10)(
    delayed(get_bikeability_igraph)(r_gts, G_base, od_demand, exit_points)
    for r_gts in random_gts_list_sampled)
mean_random_continuous = np.mean(random_continuous_all, axis=0)



plt.figure(figsize=(10, 6))
step_size = max(1, round(len(demand_gts) / len(demand_continuous_scores)))
x_axis = np.arange(len(demand_continuous_scores)) * step_size


for run_scores in random_continuous_all:
    plt.plot(x_axis, run_scores, color='blue', alpha=0.15, linewidth=1)
plt.plot(x_axis, mean_random_continuous, color='blue', linestyle='--', linewidth=2.5, label='Random Mean')
plt.plot(x_axis, demand_continuous_scores, color='red', linewidth=3, linestyle='--', label='Demand Weighted')
plt.plot(x_axis, demand_ltn_priority_continuous_scores, color='green', linewidth=3, linestyle='-', label='Demand-LTN Priority')
plt.plot(x_axis, betweenness_ltn_priority_continuous_scores, color='orange', linestyle='-', linewidth=3, label='Betweenness-LTN Priority')
plt.plot(x_axis, betweenness_continuous_scores, color='purple', linewidth=3, linestyle='-.', label='Betweenness')

plt.title('Benefit to cyclist in reduction of "perceived distance"')
plt.xlabel('Growth Stage')
plt.ylabel('Average Savings of perceived distance (meters)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\gateshead_bikeability_continuous_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

------

## Experimenting with the influence of demand on the benefit metric

compare three models: Rule of Half, Exponential, Logarthimic

In [ ]:
import igraph as ig
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

def get_bikeability_weighted(results_gts, G_base, od_demand, exit_points, 
                             penalty_cost=20000, run_id="Main", 
                             weighting_mode="linear", k=2):
    # 1. SETUP IGRAPH
    g_base_ig = ig.Graph.from_networkx(G_base.to_undirected())
    
    # CRITICAL: Ensure every single edge has a valid weight. 
    # If it's missing or NaN, we set it to the penalty_cost (effectively impassable)
    weights = []
    for e in g_base_ig.es:
        w = e.attributes().get('lts_length')
        weights.append(w if (w is not None and not np.isnan(w)) else penalty_cost)
    g_base_ig.es['lts_length'] = weights

    osmid_to_idx = {osmid: idx for idx, osmid in enumerate(g_base_ig.vs['_nx_name'])}
    
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(
        lambda x: [osmid_to_idx[o] for o in x if o in osmid_to_idx]
    ).to_dict()

    # 2. PREPARE TRIPS
    trips = []
    for _, row in od_demand.iterrows():
        if pd.isna(row['start_osmid']) or row['total_flow'] <= 0: continue
        
        s_idxs = exit_dict.get(int(row['start_neighbourhood_id']), []) if row['ltn_origin'] else []
        if not s_idxs and row['start_osmid'] in osmid_to_idx: 
            s_idxs = [osmid_to_idx[row['start_osmid']]]
            
        e_idxs = exit_dict.get(int(row['end_neighbourhood_id']), []) if row['ltn_destination'] else []
        if not e_idxs and row['end_osmid'] in osmid_to_idx: 
            e_idxs = [osmid_to_idx[row['end_osmid']]]
        
        # Only add trip if it has at least one valid start and end node
        if s_idxs and e_idxs:
            trips.append({'starts': s_idxs, 'ends': e_idxs, 'flow': row['total_flow']})

    if not trips: 
        print(f"Warning: No valid trips found for {run_id}")
        return [0.0] * len(results_gts)

    # 3. ROBUST COST CALCULATION
    def compute_costs(graph):
        all_costs = []
        for t in trips:
            # igraph returns a list of lists. If a path doesn't exist, value is 'inf'
            d = graph.distances(source=t['starts'], target=t['ends'], weights='lts_length')
            # Flatten and find min
            min_dist = np.min(d)
            all_costs.append(min(min_dist, penalty_cost))
        return np.array(all_costs)

    print(f"[{run_id}] Calculating Baseline for {len(trips)} trips...")
    base_costs = compute_costs(g_base_ig)
    stage_results = []
    flows = np.array([t['flow'] for t in trips])

    # Pre-calculate Weights for Flow
    if weighting_mode == "exponential":
        w_demand = flows ** k
    elif weighting_mode == "logarithmic":
        w_demand = np.log1p(flows)
    else:
        w_demand = flows

    # 4. RUN STAGES
    for i, G_routed in enumerate(results_gts):
        g_current = g_base_ig.copy()
        
        # Only add edges if G_routed is not empty/None
        if G_routed is not None and len(G_routed.edges) > 0:
            new_edges = []
            new_weights = []
            for u, v, key, d in G_routed.edges(keys=True, data=True):
                if u in osmid_to_idx and v in osmid_to_idx:
                    new_edges.append((osmid_to_idx[u], osmid_to_idx[v]))
                    # Handle potential NaN lengths in the new infrastructure
                    length = d.get('length', 0)
                    new_weights.append(length if (length is not None and not np.isnan(length)) else 0)
            
            if new_edges:
                old_ecount = g_current.ecount()
                g_current.add_edges(new_edges)
                for idx, w in enumerate(new_weights):
                    g_current.es[old_ecount + idx]['lts_length'] = w

        current_costs = compute_costs(g_current)
        savings = np.maximum(0, base_costs - current_costs)

        if weighting_mode == "rule_of_half":
            induced_demand = flows * (1 + (savings / 100) * 0.01)
            total_benefit = np.sum(savings * ((flows + induced_demand) / 2)) / np.sum((flows + induced_demand) / 2)
        else:
            total_benefit = np.sum(savings * w_demand) / np.sum(w_demand)
            
        stage_results.append(total_benefit)
        print(f"  [{run_id}] Stage {i} | {weighting_mode} Benefit: {total_benefit:.1f}")

    return stage_results

In [ ]:
# Slicing for a quick test
test_gts = demand_gts[::10]

# Calculate all three
res_linear = get_bikeability_weighted(test_gts, G_base, od_demand, exit_points, weighting_mode="linear")
res_expo = get_bikeability_weighted(test_gts, G_base, od_demand, exit_points, weighting_mode="exponential", k=1.5)
res_log = get_bikeability_weighted(test_gts, G_base, od_demand, exit_points, weighting_mode="logarithmic")

# Plotting the "Battle of the Weights"
plt.figure(figsize=(10, 5))
plt.plot(res_linear, label="Linear (Standard)", linewidth=2)
plt.plot(res_expo, label="Exponential (Efficiency - Power 1.5)", linewidth=2)
plt.plot(res_log, label="Logarithmic (Social Equity)", linewidth=2)
plt.title("How Weighting Changes the Perceived Benefit of the Same Strategy")
plt.ylabel("Weighted Avg Savings (Meters)")
plt.legend()
plt.show()

In [ ]:
# --- SETTINGS ---
MY_MODE = "linear"  # Options: "linear", "exponential", "logarithmic", "rule_of_half"
STEP = 5           # Slicing: every 10th stage
NUM_RANDOM = 5      # 5 random runs for the check

# --- PREPARE DATA ---
test_scenarios = {
    "Demand-Weighted": demand_gts[::STEP],
    "Betweenness": betweenness_gts[::STEP],
    "Demand-LTN": demand_ltn_priority_gts[::STEP],
    "Betweenness-LTN": betweenness_ltn_priority_gts[::STEP]
}

# --- 1. RUN MAIN SCENARIOS ---
results = {}
for name, gts in test_scenarios.items():
    print(f"\nEvaluating {name}...")
    results[name] = get_bikeability_weighted(
        gts, G_base, od_demand, exit_points, 
        weighting_mode=MY_MODE, run_id=name
    )

# --- 2. RUN RANDOM RUNS (PARALLEL) ---
print(f"\nRunning {NUM_RANDOM} Random runs in parallel...")
# We slice each random run in the list
random_slices = [r[::STEP] for r in random_gts_list[:NUM_RANDOM]]

random_results = Parallel(n_jobs=-2)(
    delayed(get_bikeability_weighted)(
        rg, G_base, od_demand, exit_points, 
        weighting_mode=MY_MODE, run_id=f"Rand_{i}"
    ) for i, rg in enumerate(random_slices)
)

mean_random = np.mean(random_results, axis=0)

# --- 3. PLOT EVERYTHING ---
plt.figure(figsize=(12, 7))
x_axis = np.arange(len(next(iter(results.values())))) * STEP

# Plot Individual Random Runs
for r_res in random_results:
    plt.plot(x_axis, r_res, color='blue', alpha=0.15, linewidth=1)

# Plot Averages/Strategies
plt.plot(x_axis, mean_random, color='blue', linestyle='--', label='Random (Mean)', linewidth=2.5)
plt.plot(x_axis, results["Demand-Weighted"], color='red', label='Demand-Weighted', linewidth=3)
plt.plot(x_axis, results["Betweenness"], color='purple', label='Betweenness', linewidth=3)
plt.plot(x_axis, results["Demand-LTN"], color='green', label='Demand-LTN Priority', linewidth=3)
plt.plot(x_axis, results["Betweenness-LTN"], color='orange', label='Betweenness-LTN Priority', linewidth=3)

plt.title(f'Continuous Improvement Comparison (Weighting: {MY_MODE.upper()})')
plt.xlabel('Infrastructure Growth Stage')
plt.ylabel('Weighted Avg Savings (Meters)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- SETTINGS ---
MY_MODE = "exponential"  # Options: "linear", "exponential", "logarithmic", "rule_of_half"
STEP = 5          # Slicing: every 10th stage
NUM_RANDOM = 5      # 5 random runs for the check

# --- PREPARE DATA ---
test_scenarios = {
    "Demand-Weighted": demand_gts[::STEP],
    "Betweenness": betweenness_gts[::STEP],
    "Demand-LTN": demand_ltn_priority_gts[::STEP],
    "Betweenness-LTN": betweenness_ltn_priority_gts[::STEP]
}

# --- 1. RUN MAIN SCENARIOS ---
results = {}
for name, gts in test_scenarios.items():
    print(f"\nEvaluating {name}...")
    results[name] = get_bikeability_weighted(
        gts, G_base, od_demand, exit_points, 
        weighting_mode=MY_MODE, run_id=name
    )

# --- 2. RUN RANDOM RUNS (PARALLEL) ---
print(f"\nRunning {NUM_RANDOM} Random runs in parallel...")
# We slice each random run in the list
random_slices = [r[::STEP] for r in random_gts_list[:NUM_RANDOM]]

random_results = Parallel(n_jobs=-2)(
    delayed(get_bikeability_weighted)(
        rg, G_base, od_demand, exit_points, 
        weighting_mode=MY_MODE, run_id=f"Rand_{i}"
    ) for i, rg in enumerate(random_slices)
)

mean_random = np.mean(random_results, axis=0)

# --- 3. PLOT EVERYTHING ---
plt.figure(figsize=(12, 7))
x_axis = np.arange(len(next(iter(results.values())))) * STEP

# Plot Individual Random Runs
for r_res in random_results:
    plt.plot(x_axis, r_res, color='blue', alpha=0.15, linewidth=1)

# Plot Averages/Strategies
plt.plot(x_axis, mean_random, color='blue', linestyle='--', label='Random (Mean)', linewidth=2.5)
plt.plot(x_axis, results["Demand-Weighted"], color='red', label='Demand-Weighted', linewidth=3)
plt.plot(x_axis, results["Betweenness"], color='purple', label='Betweenness', linewidth=3)
plt.plot(x_axis, results["Demand-LTN"], color='green', label='Demand-LTN Priority', linewidth=3)
plt.plot(x_axis, results["Betweenness-LTN"], color='orange', label='Betweenness-LTN Priority', linewidth=3)

plt.title(f'Continuous Improvement Comparison (Weighting: {MY_MODE.upper()})')
plt.xlabel('Infrastructure Growth Stage')
plt.ylabel('Weighted Avg Savings (Meters)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- SETTINGS ---
MY_MODE = "logarithmic"  # Options: "linear", "exponential", "logarithmic", "rule_of_half"
STEP = 5           # Slicing: every 10th stage
NUM_RANDOM = 5      # 5 random runs for the check

# --- PREPARE DATA ---
test_scenarios = {
    "Demand-Weighted": demand_gts[::STEP],
    "Betweenness": betweenness_gts[::STEP],
    "Demand-LTN": demand_ltn_priority_gts[::STEP],
    "Betweenness-LTN": betweenness_ltn_priority_gts[::STEP]
}

# --- 1. RUN MAIN SCENARIOS ---
results = {}
for name, gts in test_scenarios.items():
    print(f"\nEvaluating {name}...")
    results[name] = get_bikeability_weighted(
        gts, G_base, od_demand, exit_points, 
        weighting_mode=MY_MODE, run_id=name
    )

# --- 2. RUN RANDOM RUNS (PARALLEL) ---
print(f"\nRunning {NUM_RANDOM} Random runs in parallel...")
# We slice each random run in the list
random_slices = [r[::STEP] for r in random_gts_list[:NUM_RANDOM]]

random_results = Parallel(n_jobs=-2)(
    delayed(get_bikeability_weighted)(
        rg, G_base, od_demand, exit_points, 
        weighting_mode=MY_MODE, run_id=f"Rand_{i}"
    ) for i, rg in enumerate(random_slices)
)

mean_random = np.mean(random_results, axis=0)

# --- 3. PLOT EVERYTHING ---
plt.figure(figsize=(12, 7))
x_axis = np.arange(len(next(iter(results.values())))) * STEP

# Plot Individual Random Runs
for r_res in random_results:
    plt.plot(x_axis, r_res, color='blue', alpha=0.15, linewidth=1)

# Plot Averages/Strategies
plt.plot(x_axis, mean_random, color='blue', linestyle='--', label='Random (Mean)', linewidth=2.5)
plt.plot(x_axis, results["Demand-Weighted"], color='red', label='Demand-Weighted', linewidth=3)
plt.plot(x_axis, results["Betweenness"], color='purple', label='Betweenness', linewidth=3)
plt.plot(x_axis, results["Demand-LTN"], color='green', label='Demand-LTN Priority', linewidth=3)
plt.plot(x_axis, results["Betweenness-LTN"], color='orange', label='Betweenness-LTN Priority', linewidth=3)

plt.title(f'Continuous Improvement Comparison (Weighting: {MY_MODE.upper()})')
plt.xlabel('Infrastructure Growth Stage')
plt.ylabel('Weighted Avg Savings (Meters)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import igraph as ig
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

def get_bikeability_roh_decay(results_gts, G_base, od_demand, exit_points, 
                             penalty_cost=20000, run_id="Main", tolerance_pct=0.20):
    """
    Evaluates network quality using Rule of Half (RoH) with Dynamic Thresholds.
    Realization uses an Exponential Decay: rho = exp(-beta * extra_stress).
    Tolerance: Users accept 50% extra stress over their shortest physical path.
    """
    # --- 1. SETUP IGRAPH ---
    g_base_ig = ig.Graph.from_networkx(G_base.to_undirected())
    
    # Ensure baseline edges have valid weights
    for e in g_base_ig.es:
        lts_w = e.attributes().get('lts_length')
        phys_w = e.attributes().get('length')
        e['lts_length'] = lts_w if (lts_w is not None and not np.isnan(lts_w)) else penalty_cost
        e['length'] = phys_w if (phys_w is not None and not np.isnan(phys_w)) else 0

    osmid_to_idx = {osmid: idx for idx, osmid in enumerate(g_base_ig.vs['_nx_name'])}
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(
        lambda x: [osmid_to_idx[o] for o in x if o in osmid_to_idx]
    ).to_dict()

    # --- 2. PREPARE TRIPS & CALCULATE 'DUTCH IDEALS' ---
    trips = []
    print(f"[{run_id}] Calculating Dutch Ideals (Shortest Physical Paths)...")
    for _, row in od_demand.iterrows():
        if pd.isna(row['start_osmid']) or row['total_flow'] <= 0: continue
        
        s_idxs = exit_dict.get(int(row['start_neighbourhood_id']), []) if row['ltn_origin'] else []
        if not s_idxs and row['start_osmid'] in osmid_to_idx: s_idxs = [osmid_to_idx[row['start_osmid']]]
        e_idxs = exit_dict.get(int(row['end_neighbourhood_id']), []) if row['ltn_destination'] else []
        if not e_idxs and row['end_osmid'] in osmid_to_idx: e_idxs = [osmid_to_idx[row['end_osmid']]]
        
        if s_idxs and e_idxs:
            # Find the absolute shortest distance regardless of stress
            d_ideal = np.min(g_base_ig.distances(source=s_idxs, target=e_idxs, weights='length'))
            trips.append({
                'starts': s_idxs, 
                'ends': e_idxs, 
                'potential_flow': row['total_flow'],
                'threshold': d_ideal * (1 + tolerance_pct) # Budget = physical dist + 50%
            })

    if not trips: return [0.0] * len(results_gts)

    # --- 3. HELPERS: COSTS & EXPONENTIAL DECAY REALIZATION ---
    def compute_stage_data(graph):
        costs = []
        realized_demands = []
        beta = 0.002 # Decay parameter: controls how quickly people give up
        
        for t in trips:
            d = graph.distances(source=t['starts'], target=t['ends'], weights='lts_length')
            min_dist = min(np.min(d), penalty_cost)
            
            # EXPONENTIAL DECAY LOGIC
            # realization is 1.0 if under budget, otherwise decays
            extra_stress = max(0, min_dist - t['threshold'])
            rho = np.exp(-beta * extra_stress)
            
            costs.append(min_dist)
            realized_demands.append(t['potential_flow'] * rho)
            
        return np.array(costs), np.array(realized_demands)

    # --- 4. RUN SIMULATION ---
    print(f"[{run_id}] Calculating Baseline for {len(trips)} trips...")
    base_costs, base_demands = compute_stage_data(g_base_ig)
    potential_flows = np.array([t['potential_flow'] for t in trips])
    total_potential = np.sum(potential_flows)
    
    stage_results = []

    for i, G_routed in enumerate(results_gts):
        g_current = g_base_ig.copy()
        if G_routed is not None and len(G_routed.edges) > 0:
            new_edges, new_weights = [], []
            for u, v, key, d in G_routed.edges(keys=True, data=True):
                if u in osmid_to_idx and v in osmid_to_idx:
                    new_edges.append((osmid_to_idx[u], osmid_to_idx[v]))
                    w = d.get('length', 0)
                    new_weights.append(w if (w is not None and not np.isnan(w)) else 0)
            
            if new_edges:
                old_ecount = g_current.ecount()
                g_current.add_edges(new_edges)
                for idx, w in enumerate(new_weights):
                    g_current.es[old_ecount + idx]['lts_length'] = w

        current_costs, current_demands = compute_stage_data(g_current)
        
        # RULE OF HALF: Benefit = (C_base - C_stage) * ((D_base + D_stage) / 2)
        savings = np.maximum(0, base_costs - current_costs)
        avg_demand = (base_demands + current_demands) / 2
        total_benefit = np.sum(savings * avg_demand) / total_potential
        
        stage_results.append(total_benefit)
        print(f"  [{run_id}] Stage {i} | RoH Benefit: {total_benefit:.1f} | Realized: {np.sum(current_demands):.0f}/{total_potential:.0f}")

    return stage_results

# --- EXECUTION BLOCK ---
STEP = 10
NUM_RANDOM = 5

print("--- RUNNING DYNAMIC DECAY RoH ---")

# Main Scenarios
scenarios = {
    "Demand-Weighted": demand_gts[::STEP],
    "Betweenness": betweenness_gts[::STEP],
    "Demand-LTN": demand_ltn_priority_gts[::STEP],
    "Betweenness-LTN": betweenness_ltn_priority_gts[::STEP]
}

final_results = {}
for name, gts in scenarios.items():
    final_results[name] = get_bikeability_roh_decay(gts, G_base, od_demand, exit_points, run_id=name)

# Random Scenarios (Parallel)
print(f"\nRunning {NUM_RANDOM} Random runs in parallel...")
random_slices = [r[::STEP] for r in random_gts_list[:NUM_RANDOM]]
random_results = Parallel(n_jobs=-2)(
    delayed(get_bikeability_roh_decay)(rg, G_base, od_demand, exit_points, 20000, f"Rand_{i}")
    for i, rg in enumerate(random_slices)
)
mean_random = np.mean(random_results, axis=0)

# --- PLOTTING ---
plt.figure(figsize=(12, 7))
x_axis = np.arange(len(next(iter(final_results.values())))) * STEP

for r_res in random_results:
    plt.plot(x_axis, r_res, color='blue', alpha=0.15, linewidth=1)

plt.plot(x_axis, mean_random, color='blue', linestyle='--', label='Random (Mean)', linewidth=2.5)
plt.plot(x_axis, final_results["Demand-Weighted"], color='red', label='Demand-Weighted', linewidth=3)
plt.plot(x_axis, final_results["Betweenness"], color='purple', label='Betweenness', linewidth=3)
plt.plot(x_axis, final_results["Demand-LTN"], color='green', label='Demand-LTN Priority', linewidth=3)
plt.plot(x_axis, final_results["Betweenness-LTN"], color='orange', label='Betweenness-LTN Priority', linewidth=3)

plt.title('Welfare Benefit: Rule of Half with Exponential Decay Realization')
plt.xlabel('Infrastructure Growth Stage')
plt.ylabel('Societal Benefit (Weighted Stress-Meters Saved)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

-----

try without demand included

In [ ]:
import igraph as ig
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

def get_bikeability_utility_igraph(results_gts, G_base, od_demand, exit_points, 
                                 threshold_meters=5000, run_id="Main"):
    """
    Calculates Average Individual Utility Gain.
    Ignores demand weights to see how the average OD pair improves.
    Caps costs at threshold_meters to capture the 'becoming bikeable' jump.
    """
    # 1. SETUP IGRAPH
    g_base_ig = ig.Graph.from_networkx(G_base.to_undirected())
    osmid_to_idx = {osmid: idx for idx, osmid in enumerate(g_base_ig.vs['_nx_name'])}
    
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(
        lambda x: [osmid_to_idx[o] for o in x if o in osmid_to_idx]
    ).to_dict()

    # 2. PREPARE TRIPS (Nodes converted to igraph indices)
    trips = []
    for _, row in od_demand.iterrows():
        if pd.isna(row['start_osmid']): continue
            
        s_idxs = exit_dict.get(int(row['start_neighbourhood_id']), []) if row['ltn_origin'] else []
        if not s_idxs and row['start_osmid'] in osmid_to_idx:
            s_idxs = [osmid_to_idx[row['start_osmid']]]
            
        e_idxs = exit_dict.get(int(row['end_neighbourhood_id']), []) if row['ltn_destination'] else []
        if not e_idxs and row['end_osmid'] in osmid_to_idx:
            e_idxs = [osmid_to_idx[row['end_osmid']]]
            
        if s_idxs and e_idxs:
            trips.append({'starts': s_idxs, 'ends': e_idxs})

    if not trips: return [0.0] * len(results_gts)

    # 3. HELPER: CAPPED COST CALCULATION
    def compute_capped_costs(graph):
        costs = []
        for t in trips:
            # Get shortest path distance
            dists = graph.distances(source=t['starts'], target=t['ends'], weights='lts_length')
            min_dist = np.min(dists)
            # Cap at threshold: if it's 50km, we treat it as 15km.
            costs.append(min(min_dist, threshold_meters))
        return np.array(costs)

    # 4. RUN SIMULATION
    print(f"[{run_id}] Calculating Baseline for {len(trips)} OD pairs...")
    base_costs = compute_capped_costs(g_base_ig)
    stage_results = []

    for i, G_routed in enumerate(results_gts):
        g_current = g_base_ig.copy()
        
        if G_routed is not None and len(G_routed.edges) > 0:
            new_edges = []
            new_weights = []
            for u, v, k, d in G_routed.edges(keys=True, data=True):
                if u in osmid_to_idx and v in osmid_to_idx:
                    new_edges.append((osmid_to_idx[u], osmid_to_idx[v]))
                    new_weights.append(d.get('length', 0))
            
            g_current.add_edges(new_edges)
            start_idx = g_base_ig.ecount()
            for idx, w in enumerate(new_weights):
                g_current.es[start_idx + idx]['lts_length'] = w

        current_costs = compute_capped_costs(g_current)
        
        # Savings represents the utility gain (Enfranchisement + Optimization)
        individual_savings = base_costs - current_costs
        avg_utility_gain = np.mean(individual_savings)
        
        stage_results.append(avg_utility_gain)
        print(f"  [{run_id}] Stage {i} | Avg Utility Gain: {avg_utility_gain:.1f}m")

    return stage_results

# --- EXECUTION BLOCK ---

# Slicing for speed (Every 5th stage, first 10 random runs)
step = 20
num_random = 10

d_gts = demand_gts[::step]
dl_gts = demand_ltn_priority_gts[::step]
bl_gts = betweenness_ltn_priority_gts[::step]
b_gts = betweenness_gts[::step]
r_gts_list = [r[::step] for r in random_gts_list[:num_random]]

# Run Main Scenarios
scenarios = {
    "Demand": d_gts,
    "Demand-LTN": dl_gts,
    "Betweenness": b_gts,
    "Betweenness-LTN": bl_gts
}

threshold_meters = 5000

results = {}
for name, gts in scenarios.items():
    results[name] = get_bikeability_utility_igraph(gts, G_base, od_demand, exit_points, threshold_meters, run_id=name)

# Run Random Scenarios in Parallel
print(f"\nRunning {num_random} Random runs in parallel...")
random_results = Parallel(n_jobs=-2)(
    delayed(get_bikeability_utility_igraph)(rg, G_base, od_demand, exit_points, threshold_meters, f"Rand_{i}")
    for i, rg in enumerate(r_gts_list)
)
mean_random = np.mean(random_results, axis=0)

# --- PLOTTING ---
plt.figure(figsize=(12, 7))
x_axis = np.arange(len(d_gts)) * step

for r_res in random_results:
    plt.plot(x_axis, r_res, color='blue', alpha=0.1, linewidth=1)

plt.plot(x_axis, mean_random, color='blue', linestyle='--', label='Random (Mean)', linewidth=2)
plt.plot(x_axis, results["Demand"], color='red', label='Demand Weighted', linewidth=3)
plt.plot(x_axis, results["Demand-LTN"], color='green', label='Demand-LTN Priority', linewidth=3)
plt.plot(x_axis, results["Betweenness"], color='purple', label='Betweenness', linewidth=3)
plt.plot(x_axis, results["Betweenness-LTN"], color='orange', label='Betweenness-LTN Priority', linewidth=3)

plt.title('Network Utility Growth: Enfranchisement & Optimization Combined')
plt.xlabel('Infrastructure Growth Stage')
plt.ylabel('Average Utility Gained (Meters of Perceived Distance)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
x_axis = np.arange(len(d_gts)) * step

for r_res in random_results:
    plt.plot(x_axis, r_res, color='blue', alpha=0.1, linewidth=1)

plt.plot(x_axis, mean_random, color='blue', linestyle='--', label='Random (Mean)', linewidth=2)
plt.plot(x_axis, results["Demand"], color='red', label='Demand Weighted', linewidth=3)
plt.plot(x_axis, results["Demand-LTN"], color='green', label='Demand-LTN Priority', linewidth=3)
plt.plot(x_axis, results["Betweenness"], color='purple', label='Betweenness', linewidth=3)
plt.plot(x_axis, results["Betweenness-LTN"], color='orange', label='Betweenness-LTN Priority', linewidth=3)

plt.title('Average LTS Distance between OD pairs improvement with network growth (Capped at 5000m)')
plt.xlabel('Infrastructure Growth Stage')
plt.ylabel('Average number of meters of Perceived Distance reduced')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\gateshead_bikeability_utility_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

--------------------------

----------------

---------------------------------

# Make GISRUK plots!

Rather than having to wait for notebook 05, I'm making some plots to share at GISRUK here.

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import osmnx as ox
import igraph as ig
import matplotlib.pyplot as plt
import gc

# ==========================================
# 1. METRIC FUNCTIONS (Using your exact logic)
# ==========================================

def get_demand_met_fast(G, trips_dict):
    if not G or not hasattr(G, 'edges'):
        return 0.0
    unique_edges = set(tuple(sorted((u, v))) for u, v in G.edges())
    return sum(trips_dict.get(edge, 0.0) for edge in unique_edges)

def evaluate_demand_met(graphs_list, trips_dict):
    return [get_demand_met_fast(G, trips_dict) for G in graphs_list]

def load_scenario_data(filepath):
    if not os.path.exists(filepath):
        return [], []
    with open(filepath, "rb") as f:
        data = pickle.load(f)
    gts = data.get("GTs", [])
    abstracts = data.get("GT_abstracts", [])
    gts.insert(0, nx.MultiGraph())
    abstracts.insert(0, nx.MultiGraph())
    return gts, abstracts

def get_bikeablity_lts_2_only(results_gts, G_LTS_2, od_demand, exit_points):
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(list).to_dict()
    processed_trips = []
    for s_node, e_node, flow, is_ltn_o, is_ltn_d, s_hood, e_hood in zip(
        od_demand['start_osmid'], od_demand['end_osmid'], od_demand['total_flow'],
        od_demand['ltn_origin'], od_demand['ltn_destination'],
        od_demand['start_neighbourhood_id'], od_demand['end_neighbourhood_id'] ):
        if pd.isna(s_node) or pd.isna(e_node):
            continue  
            
        start_targets = [int(s_node)]
        end_targets = [int(e_node)]
        
        if is_ltn_o and pd.notna(s_hood) and int(s_hood) in exit_dict:
            start_targets = exit_dict[int(s_hood)]
                
        if is_ltn_d and pd.notna(e_hood) and int(e_hood) in exit_dict:
            end_targets = exit_dict[int(e_hood)]
            
        processed_trips.append({
            'starts': start_targets, 'ends': end_targets,
            'flow': flow if pd.notna(flow) else 0
        })

    G_base_undir = G_LTS_2.to_undirected()
    bikeable_flows_per_stage = []

    for i, G_routed in enumerate(results_gts):
        G_current = G_base_undir.copy()
        if G_routed is not None:
            G_current.add_edges_from(G_routed.edges())
        
        node_to_neighbourhood = {}
        for island_id, component_nodes in enumerate(nx.connected_components(G_current)):
            for node in component_nodes:
                node_to_neighbourhood[node] = island_id

        stage_total_flow = 0
        for trip in processed_trips:
            start_neighbourhoods = set(node_to_neighbourhood[s] for s in trip['starts'] if s in node_to_neighbourhood)
            end_neighbourhoods = set(node_to_neighbourhood[e] for e in trip['ends'] if e in node_to_neighbourhood)
            
            if not start_neighbourhoods or not end_neighbourhoods:
                continue
            if start_neighbourhoods.intersection(end_neighbourhoods):
                stage_total_flow += trip['flow']
                
        bikeable_flows_per_stage.append(stage_total_flow)
        
    return bikeable_flows_per_stage

def get_bikeablity_lts_length_cutoff(results_gts, G_base, od_demand, exit_points, cutoff=5000):
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(list).to_dict()
    processed_trips = []
    for s_node, e_node, flow, is_ltn_o, is_ltn_d, s_hood, e_hood in zip(
        od_demand['start_osmid'], od_demand['end_osmid'], od_demand['total_flow'],
        od_demand['ltn_origin'], od_demand['ltn_destination'],
        od_demand['start_neighbourhood_id'], od_demand['end_neighbourhood_id'] ):
        if pd.isna(s_node) or pd.isna(e_node):
            continue  
            
        start_targets = [int(s_node)]
        end_targets = [int(e_node)]
        
        if is_ltn_o and pd.notna(s_hood) and int(s_hood) in exit_dict:
            start_targets = exit_dict[int(s_hood)]
                
        if is_ltn_d and pd.notna(e_hood) and int(e_hood) in exit_dict:
            end_targets = exit_dict[int(e_hood)]
            
        processed_trips.append({
            'starts': start_targets, 'ends': end_targets,
            'flow': flow if pd.notna(flow) else 0
        })
    
    G_base_undir = G_base.to_undirected() 
    bikeable_flows_per_stage = []

    for i, G_routed in enumerate(results_gts):
        G_current = G_base_undir.copy()
        
        if G_routed is not None:
            new_edges_formatted = []
            for u, v, k, d in G_routed.edges(keys=True, data=True):
                edge_length = d.get('length', 0)
                d['lts_class'] = 1
                d['lts_length'] = edge_length * 1 
                new_edges_formatted.append((u, v, k, d))
            G_current.add_edges_from(new_edges_formatted)
        
        stage_total_flow = 0
        for trip in processed_trips:
            valid_starts = [s for s in trip['starts'] if G_current.has_node(s)]
            valid_ends = set(e for e in trip['ends'] if G_current.has_node(e))
            
            if not valid_starts or not valid_ends:
                continue
                
            try:
                lengths = nx.multi_source_dijkstra_path_length(
                    G_current, valid_starts, cutoff=cutoff, weight='lts_length'
                )
                reached_ends = valid_ends.intersection(lengths.keys())
                if reached_ends:
                    stage_total_flow += trip['flow']
            except nx.NodeNotFound:
                pass

        bikeable_flows_per_stage.append(stage_total_flow)
        
    return bikeable_flows_per_stage

def get_bikeability_igraph(results_gts, G_base, od_demand, exit_points, penalty_cost=20000):
    g_base_ig = ig.Graph.from_networkx(G_base.to_undirected())
    osmid_to_idx = {osmid: idx for idx, osmid in enumerate(g_base_ig.vs['_nx_name'])}
    
    exit_dict = exit_points.groupby('neighbourhood_id')['osmid'].apply(
        lambda x: [osmid_to_idx[o] for o in x if o in osmid_to_idx]
    ).to_dict()

    trips = []
    total_demand = 0.0
    for _, row in od_demand.iterrows():
        if pd.isna(row['start_osmid']) or row['total_flow'] <= 0:
            continue
            
        s_idxs = exit_dict.get(int(row['start_neighbourhood_id']), []) if row['ltn_origin'] else []
        if not s_idxs and row['start_osmid'] in osmid_to_idx:
            s_idxs = [osmid_to_idx[row['start_osmid']]]
            
        e_idxs = exit_dict.get(int(row['end_neighbourhood_id']), []) if row['ltn_destination'] else []
        if not e_idxs and row['end_osmid'] in osmid_to_idx:
            e_idxs = [osmid_to_idx[row['end_osmid']]]
            
        if s_idxs and e_idxs:
            trips.append({'starts': s_idxs, 'ends': e_idxs, 'flow': row['total_flow']})
            total_demand += row['total_flow']

    if total_demand == 0: return [0.0] * len(results_gts)

    def compute_stage_costs(graph):
        costs = []
        for t in trips:
            dists = graph.distances(source=t['starts'], target=t['ends'], weights='lts_length')
            min_dist = np.min(dists)
            costs.append(min_dist if min_dist < penalty_cost else penalty_cost)
        return np.array(costs)

    base_costs = compute_stage_costs(g_base_ig)
    stage_results = []

    for i, G_routed in enumerate(results_gts):
        g_current = g_base_ig.copy()
        
        if G_routed is not None and len(G_routed.edges) > 0:
            new_edges = []
            new_weights = []
            for u, v, k, d in G_routed.edges(keys=True, data=True):
                if u in osmid_to_idx and v in osmid_to_idx:
                    new_edges.append((osmid_to_idx[u], osmid_to_idx[v]))
                    new_weights.append(d.get('length', 0))
            
            g_current.add_edges(new_edges)
            start_idx = g_base_ig.ecount()
            for idx, w in enumerate(new_weights):
                g_current.es[start_idx + idx]['lts_length'] = w

        current_costs = compute_stage_costs(g_current)
        savings = np.maximum(0, base_costs - current_costs)
        avg_savings = np.sum(savings * [t['flow'] for t in trips]) / total_demand
        stage_results.append(avg_savings)

    return stage_results

def plot_metric(stages, scenario_dict, random_lists, title, ylabel, save_path):
    plt.figure(figsize=(10, 6))
    for i, r_flows in enumerate(random_lists):
        label = 'Random (Individual)' if i == 0 else None
        plt.plot(stages, r_flows, color='blue', alpha=0.15, linewidth=1.5, label=label)

    if random_lists:
        plt.plot(stages, np.mean(random_lists, axis=0), color='blue', linestyle='--', linewidth=2.5, label='Random (Mean)')

    colors = {'Demand': 'red', 'Betweenness': 'orange', 'Demand LTN Priority': 'green', 'Betweenness LTN Priority': 'purple'}
    
    for name, data in scenario_dict.items():
        plt.plot(stages[:len(data)], data, color=colors.get(name, 'black'), linestyle='-', linewidth=2.5, label=name)

    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Infrastructure Added (Stages)", fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# ==========================================
# 2. MAIN EXECUTION LOOP
# ==========================================

places = ["gateshead", "newcastle", "north_tyneside", "south_tyneside", "sunderland"]
base_dir = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external"
debug_dir = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging"
os.makedirs(debug_dir, exist_ok=True)

lts_mapping = {
    "motorway": 4, "motorway_link": 4, "trunk": 4, "trunk_link": 4,
    "primary": 4, "primary_link": 4, "secondary": 4, "secondary_link": 4,
    "tertiary": 3, "tertiary_link": 3, "unclassified": 3,
    "residential": 2, "living_street": 2,
    "cycleway": 1, "track": 1, "path": 1, "bridleway": 1, "footway": 1, "pedestrian": 1
}

STAGE_SAMPLE_RATE = 100  # Sample every time

for place in places:
    print(f"\n{'='*50}\nProcessing Location: {place.upper()}\n{'='*50}")
    
    results_dir = os.path.join(base_dir, "results", place, "current_ltn_scenario")
    data_dir = os.path.join(base_dir, "data", place, "current_ltn_scenario")
    exports_dir = os.path.join(base_dir, "exports_gpkg", place, "current_ltn_scenario")

    # ---------------------------------------------------------
    # A. Load & Prepare Spatial Data
    # ---------------------------------------------------------
    print("Loading and preparing spatial data...")
    od_demand = gpd.read_file(os.path.join(data_dir, f"{place}_current_ltn_scenario_greedy_demand_weighted.gpkg"))
    exit_points = gpd.read_file(os.path.join(exports_dir, f"{place}_exit_points.gpkg"))
    ltns = gpd.read_file(os.path.join(data_dir, f"scored_neighbourhoods_{place}.gpkg"))
    
    bikeable_gpkg = os.path.join(data_dir, f"{place}_biketrackcarall.gpkg")
    G_bikeable_nx = ox.graph_from_gdfs(
        gpd.read_file(bikeable_gpkg, layer="nodes").set_index("osmid"), 
        gpd.read_file(bikeable_gpkg, layer="edges").set_index(["u", "v", "key"])
    )

    # OD Demand Spatial Joins
    start_points = od_demand['geometry'].apply(lambda line: line.interpolate(0, normalized=True) if line is not None else None)
    end_points = od_demand['geometry'].apply(lambda line: line.interpolate(1, normalized=True) if line is not None else None)
    gdf_starts = gpd.GeoDataFrame(geometry=start_points, crs=od_demand.crs)
    gdf_ends = gpd.GeoDataFrame(geometry=end_points, crs=od_demand.crs)
    
    if ltns.crs != od_demand.crs:
        ltns = ltns.to_crs(od_demand.crs)
    ltns_clean = ltns.drop(columns=['index_left', 'index_right'], errors='ignore').copy()
    ltns_clean['neighbourhood_id'] = ltns_clean.index

    starts_joined = gpd.sjoin(gdf_starts, ltns_clean, how='left', predicate='intersects')
    ends_joined = gpd.sjoin(gdf_ends, ltns_clean, how='left', predicate='intersects')
    starts_joined = starts_joined[~starts_joined.index.duplicated(keep='first')]
    ends_joined = ends_joined[~ends_joined.index.duplicated(keep='first')]

    od_demand['start_neighbourhood_id'] = starts_joined['neighbourhood_id']
    od_demand['end_neighbourhood_id'] = ends_joined['neighbourhood_id']
    od_demand['ltn_origin'] = od_demand['start_neighbourhood_id'].notna()
    od_demand['ltn_destination'] = od_demand['end_neighbourhood_id'].notna()

    # Graph Preprocessing
    G_base = G_bikeable_nx.copy()
    _, edges_gdf = ox.graph_to_gdfs(G_base)
    if edges_gdf.crs != ltns.crs: 
        ltns = ltns.to_crs(edges_gdf.crs)
    
    edges_gdf = edges_gdf.drop(columns=['index_left', 'index_right'], errors='ignore')
    ltns = ltns.drop(columns=['index_left', 'index_right'], errors='ignore')
    edges_in_ltn = gpd.sjoin(edges_gdf, ltns, how='inner', predicate='intersects')
    ltn_edge_indices = set(edges_in_ltn.index)
    nx.set_edge_attributes(G_base, {(u, v, k): ((u, v, k) in ltn_edge_indices) for u, v, k in G_base.edges(keys=True)}, 'ltn_flag')

    for u, v, key, data in G_base.edges(keys=True, data=True):
        is_ltn = data.get('ltn_flag')
        lts_class = 1 if (is_ltn is True or str(is_ltn).lower() == 'true') else lts_mapping.get(data.get('highway', ['']) if isinstance(data.get('highway'), list) else data.get('highway'), 4)
        data['lts_class'] = lts_class
        data['lts_length'] = data.get('length', 0) * lts_class

    if G_base.is_directed(): 
        G_base = G_base.to_undirected()
        
    # Corrected G_LTS_2 logic (Keeps nodes intact)
    G_LTS_2 = G_base.copy()
    edges_to_remove = [(u, v, k) for u, v, k, d in G_LTS_2.edges(keys=True, data=True) if d.get('lts_class', 4) > 2]
    G_LTS_2.remove_edges_from(edges_to_remove)

    mydemand = od_demand.copy()
    mydemand["edge_id"] = mydemand.apply(lambda r: tuple(sorted((r.start_osmid, r.end_osmid))), axis=1)
    trips_dict = dict(zip(mydemand["edge_id"], mydemand["total_flow"]))

    # ---------------------------------------------------------
    # B. Main Scenarios Evaluation
    # ---------------------------------------------------------
    print("Evaluating main scenarios...")
    paths = {
        'Demand': os.path.join(results_dir, f"{place}_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle"),
        'Betweenness': os.path.join(results_dir, f"{place}_poi_LTNs_tessellation_betweenness_weighted_current_ltn_scenario.pickle"),
        'Demand LTN Priority': os.path.join(results_dir, f"{place}_poi_LTNs_tessellation_demand_ltn_priority_weighted_current_ltn_scenario.pickle"),
        'Betweenness LTN Priority': os.path.join(results_dir, f"{place}_poi_LTNs_tessellation_betweenness_ltn_priority_weighted_current_ltn_scenario.pickle")
    }

    results_strict, results_cutoff, results_demand_met, results_continuous = {}, {}, {}, {}
    stages, stages_sampled = None, None

    for name, path in paths.items():
        gts, abstracts = load_scenario_data(path)
        if stages is None and gts:
            stages = list(range(len(gts)))
            stages_sampled = stages[::STAGE_SAMPLE_RATE]

        if gts and abstracts:
            results_strict[name] = get_bikeablity_lts_2_only(gts, G_LTS_2, od_demand, exit_points)
            results_cutoff[name] = get_bikeablity_lts_length_cutoff(gts, G_base, od_demand, exit_points, cutoff=5000)
            results_demand_met[name] = evaluate_demand_met(abstracts, trips_dict)
            results_continuous[name] = get_bikeability_igraph(gts[::STAGE_SAMPLE_RATE], G_base, od_demand, exit_points)

        del gts, abstracts
        gc.collect()

    # ---------------------------------------------------------
    # C. Random Runs Evaluation (Streaming all files)
    # ---------------------------------------------------------
    random_files = sorted(glob.glob(os.path.join(results_dir, f"{place}_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle")))
    print(f"Streaming and evaluating all {len(random_files)} Random Runs...")
    
    random_strict_all, random_cutoff_all, random_demand_all, random_continuous_all = [], [], [], []

    for i, f in enumerate(random_files):
        print(f"  -> Random Run {i+1}/{len(random_files)}")
        r_gts, r_abs = load_scenario_data(f)
        if r_gts and r_abs:
            random_strict_all.append(get_bikeablity_lts_2_only(r_gts, G_LTS_2, od_demand, exit_points))
            random_cutoff_all.append(get_bikeablity_lts_length_cutoff(r_gts, G_base, od_demand, exit_points, cutoff=5000))
            random_demand_all.append(evaluate_demand_met(r_abs, trips_dict))
            random_continuous_all.append(get_bikeability_igraph(r_gts[::STAGE_SAMPLE_RATE], G_base, od_demand, exit_points))
            
        del r_gts, r_abs
        gc.collect()

    # ---------------------------------------------------------
    # D. Generate Output Plots
    # ---------------------------------------------------------
    print(f"Generating line plots for {place}...")
    plot_metric(stages, results_strict, random_strict_all, f"{place.title()} - Strict LTS <= 2", "Total Connected Flow", os.path.join(debug_dir, f"{place}_bikeability_strict.png"))
    plot_metric(stages, results_cutoff, random_cutoff_all, f"{place.title()} - 5000m Cutoff", "Total Connected Flow", os.path.join(debug_dir, f"{place}_bikeability_cutoff.png"))
    plot_metric(stages, results_demand_met, random_demand_all, f"{place.title()} - Demand Met", "Cycling Demand Met", os.path.join(debug_dir, f"{place}_demand_met.png"))
    plot_metric(stages_sampled, results_continuous, random_continuous_all, f"{place.title()} - Continuous (Perceived Distance)", "Avg Savings of Perceived Distance (m)", os.path.join(debug_dir, f"{place}_bikeability_continuous.png"))

    # Cleanup before next place
    del G_base, G_LTS_2, G_bikeable_nx
    gc.collect()

print("\nAll locations processed successfully.")

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

places = ["gateshead", "newcastle", "north_tyneside", "south_tyneside", "sunderland"]
debug_dir = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging"

# 1. Load the data
all_data = {}
for place in places:
    filepath = os.path.join(debug_dir, f"{place}_metrics_arrays.pickle")
    if os.path.exists(filepath):
        with open(filepath, "rb") as f:
            all_data[place] = pickle.load(f)
    else:
        print(f"[!] Missing data for {place}. Did you re-run the main cell with the save step?")

if len(all_data) == len(places):
    print("All regional data loaded successfully! Calculating overall averages...\n")

# Helper function to pad shorter arrays with their final value
def pad_to_max(arrays_list, max_len):
    padded = []
    for arr in arrays_list:
        arr_list = list(arr)
        if len(arr_list) < max_len:
            arr_list.extend([arr_list[-1]] * (max_len - len(arr_list)))
        padded.append(arr_list)
    return np.array(padded)

metric_names = [
    'Method 1: Strict LTS <= 2', 
    'Method 2: 5000m Cutoff', 
    'Method 3: Demand Met Fast', 
    'Method 4: Continuous Perceived Dist'
]

scenarios = ['Demand', 'Betweenness', 'Demand LTN Priority', 'Betweenness LTN Priority']

overall_results = {}

# 2. Process and Average each metric
for metric in metric_names:
    print(f"Aggregating {metric}...")
    
    # Find the absolute maximum number of stages across all 5 places
    max_len = max([len(all_data[p][metric]['stages']) for p in places])
    
    # Generate an overall x-axis (using the step size from the metric)
    if 'Continuous' in metric:
        step_size = 10 # Sample rate from the previous script
        overall_stages = list(range(0, max_len * step_size, step_size))
    else:
        overall_stages = list(range(max_len))
        
    overall_results[metric] = {'stages': overall_stages, 'scenarios': {}, 'random_mean': None, 'random_all': []}
    
    # Average the main scenarios
    for sc in scenarios:
        scenario_arrays = [all_data[p][metric]['scenarios'][sc] for p in places]
        padded_scenarios = pad_to_max(scenario_arrays, max_len)
        overall_results[metric]['scenarios'][sc] = np.mean(padded_scenarios, axis=0)
        
    # Aggregate and average ALL random runs (50 runs * 5 places = 250 total runs)
    all_random_runs = []
    for p in places:
        all_random_runs.extend(all_data[p][metric]['random'])
        
    padded_randoms = pad_to_max(all_random_runs, max_len)
    overall_results[metric]['random_all'] = padded_randoms
    overall_results[metric]['random_mean'] = np.mean(padded_randoms, axis=0)


# 3. Plotting the Overall Results
def plot_overall(stages, scenario_dict, random_all, random_mean, title, ylabel, save_path):
    plt.figure(figsize=(12, 7))
    
    # Plot a sample of the random runs so it isn't a solid block of blue ink
    for i, r_flows in enumerate(random_all[::5]): 
        label = 'Random (Sampled)' if i == 0 else None
        plt.plot(stages, r_flows, color='blue', alpha=0.05, linewidth=1.0, label=label)

    plt.plot(stages, random_mean, color='blue', linestyle='--', linewidth=2.5, label='Random (Overall Mean)')

    colors = {'Demand': 'red', 'Betweenness': 'orange', 'Demand LTN Priority': 'green', 'Betweenness LTN Priority': 'purple'}
    
    for name, plot_data in scenario_dict.items():
        plt.plot(stages, plot_data, color=colors.get(name, 'black'), linestyle='-', linewidth=2.5, label=name)

    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel("Infrastructure Added (Stages)", fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

print("\nGenerating final overall plots...")
plot_overall(overall_results[metric_names]['stages'], overall_results[metric_names]['scenarios'], overall_results[metric_names]['random_all'], overall_results[metric_names]['random_mean'], "OVERALL REGION - Strict LTS <= 2", "Average Connected Flow", os.path.join(debug_dir, "OVERALL_bikeability_strict.png"))
plot_overall(overall_results[metric_names]['stages'], overall_results[metric_names]['scenarios'], overall_results[metric_names]['random_all'], overall_results[metric_names]['random_mean'], "OVERALL REGION - 5000m Cutoff", "Average Connected Flow", os.path.join(debug_dir, "OVERALL_bikeability_cutoff.png"))
plot_overall(overall_results[metric_names]['stages'], overall_results[metric_names]['scenarios'], overall_results[metric_names]['random_all'], overall_results[metric_names]['random_mean'], "OVERALL REGION - Demand Met", "Average Cycling Demand Met", os.path.join(debug_dir, "OVERALL_demand_met.png"))
plot_overall(overall_results[metric_names]['stages'], overall_results[metric_names]['scenarios'], overall_results[metric_names]['random_all'], overall_results[metric_names]['random_mean'], "OVERALL REGION - Continuous (Perceived Distance)", "Average Savings (m)", os.path.join(debug_dir, "OVERALL_bikeability_continuous.png"))

print("Overall Regional analysis complete!")

## load

In [ ]:
get_demand_met_fast()

get_bikeablity_lts_2_only()

get_bikeablity_lts_length_cutoff()

#

## analyse

## plot

In [ ]:
from pathlib import Path

def tree(path, prefix="", ignore_dirs={"cache", "objects", "10_20", "GT_0", "GT_1", "GT_2", "GT_3", "GT_4", "GT_5", "GT_6", "GT_7", "GT_8", "GT_9", "GT_10"}):
    path = Path(path)
    contents = [p for p in path.iterdir() if p.name not in ignore_dirs]
    pointers = ['├── '] * (len(contents) - 1) + ['└── ']

    for pointer, p in zip(pointers, contents):
        print(prefix + pointer + p.name)
        if p.is_dir():
            extension = '│   ' if pointer == '├── ' else '    '
            tree(p, prefix + extension, ignore_dirs)

tree(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked")

-------------------------------------